In [ ]:
%gui qt


In [ ]:
# stripped notebook-only line: %gui qt

# %gui qt
from __future__ import annotations

import re, time, csv, json, datetime as dt, threading, queue
from pathlib import Path
from typing import Dict, Optional, Tuple, Any, List

import serial
import serial.tools.list_ports

# --- Built-in layout presets (used by Layout dialog)
CLASSIC_LAYOUT_PRESET_CFG = json.loads(r'''{
  "layout": {
    "cages": [
      {
        "cell_id": "3C",
        "cage_no": "2",
        "cage_type": "Food"
      },
      {
        "cell_id": "5C",
        "cage_no": "3",
        "cage_type": "Stimulus"
      },
      {
        "cell_id": "3E",
        "cage_no": "1",
        "cage_type": "Stimulus"
      },
      {
        "cell_id": "5E",
        "cage_no": "4",
        "cage_type": "Food"
      }
    ],
    "tunnels": [
      {
        "tunnel_no": 1,
        "name": "T1",
        "start_cell_id": "3C",
        "end_cell_id": "3E",
        "dead_end": false,
        "start_anchor_idx": 1,
        "end_anchor_idx": 0,
        "antenna_count": 2,
        "antennas": [
          "2",
          "1"
        ]
      },
      {
        "tunnel_no": 2,
        "name": "T2",
        "start_cell_id": "5C",
        "end_cell_id": "5E",
        "dead_end": false,
        "start_anchor_idx": 1,
        "end_anchor_idx": 0,
        "antenna_count": 2,
        "antennas": [
          "5",
          "6"
        ]
      },
      {
        "tunnel_no": 3,
        "name": "T3",
        "start_cell_id": "3C",
        "end_cell_id": "5C",
        "dead_end": false,
        "start_anchor_idx": 3,
        "end_anchor_idx": 2,
        "antenna_count": 2,
        "antennas": [
          "3",
          "4"
        ]
      },
      {
        "tunnel_no": 4,
        "name": "T4",
        "start_cell_id": "3E",
        "end_cell_id": "5E",
        "dead_end": false,
        "start_anchor_idx": 3,
        "end_anchor_idx": 2,
        "antenna_count": 2,
        "antennas": [
          "8",
          "7"
        ]
      }
    ],
    "antenna_combinations": {
      "2_1": "c2_c1",
      "1_2": "c1_c2",
      "5_6": "c3_c4",
      "6_5": "c4_c3",
      "3_4": "c2_c3",
      "4_3": "c3_c2",
      "8_7": "c1_c4",
      "7_8": "c4_c1",
      "2_2": "cage_2",
      "3_3": "cage_2",
      "2_3": "cage_2",
      "3_2": "cage_2",
      "2_4": "cage_2",
      "4_2": "cage_2",
      "3_1": "cage_2",
      "1_3": "cage_2",
      "5_5": "cage_3",
      "4_4": "cage_3",
      "5_4": "cage_3",
      "4_5": "cage_3",
      "5_3": "cage_3",
      "3_5": "cage_3",
      "4_6": "cage_3",
      "6_4": "cage_3",
      "1_1": "cage_1",
      "8_8": "cage_1",
      "1_8": "cage_1",
      "8_1": "cage_1",
      "1_7": "cage_1",
      "7_1": "cage_1",
      "8_2": "cage_1",
      "2_8": "cage_1",
      "6_6": "cage_4",
      "7_7": "cage_4",
      "6_7": "cage_4",
      "7_6": "cage_4",
      "6_8": "cage_4",
      "8_6": "cage_4",
      "7_5": "cage_4",
      "5_7": "cage_4"
    },
    "tunnels_map": {
      "c2_c1": "tunnel_1",
      "c1_c2": "tunnel_1",
      "c3_c4": "tunnel_2",
      "c4_c3": "tunnel_2",
      "c2_c3": "tunnel_3",
      "c3_c2": "tunnel_3",
      "c1_c4": "tunnel_4",
      "c4_c1": "tunnel_4"
    }
  }
}''')

FIELD_LAYOUT_PRESET_CFG = json.loads(r'''{
  "layout": {
    "cages": [
      {
        "cell_id": "2B",
        "cage_no": "3",
        "cage_type": ""
      },
      {
        "cell_id": "4B",
        "cage_no": "4",
        "cage_type": ""
      },
      {
        "cell_id": "6B",
        "cage_no": "5",
        "cage_type": ""
      },
      {
        "cell_id": "2D",
        "cage_no": "2",
        "cage_type": ""
      },
      {
        "cell_id": "6D",
        "cage_no": "6",
        "cage_type": ""
      },
      {
        "cell_id": "2F",
        "cage_no": "1",
        "cage_type": ""
      },
      {
        "cell_id": "4F",
        "cage_no": "8",
        "cage_type": ""
      },
      {
        "cell_id": "6F",
        "cage_no": "7",
        "cage_type": ""
      }
    ],
    "tunnels": [
      {
        "tunnel_no": 1,
        "name": "T1",
        "start_cell_id": "2F",
        "end_cell_id": "2D",
        "dead_end": false,
        "start_anchor_idx": 0,
        "end_anchor_idx": 1,
        "antenna_count": 2,
        "antennas": [
          "1",
          "2"
        ]
      },
      {
        "tunnel_no": 2,
        "name": "T2",
        "start_cell_id": "2D",
        "end_cell_id": "2B",
        "dead_end": false,
        "start_anchor_idx": 0,
        "end_anchor_idx": 1,
        "antenna_count": 2,
        "antennas": [
          "3",
          "4"
        ]
      },
      {
        "tunnel_no": 3,
        "name": "T3",
        "start_cell_id": "2B",
        "end_cell_id": "4B",
        "dead_end": false,
        "start_anchor_idx": 3,
        "end_anchor_idx": 2,
        "antenna_count": 2,
        "antennas": [
          "5",
          "6"
        ]
      },
      {
        "tunnel_no": 4,
        "name": "T4",
        "start_cell_id": "4B",
        "end_cell_id": "6B",
        "dead_end": false,
        "start_anchor_idx": 3,
        "end_anchor_idx": 2,
        "antenna_count": 2,
        "antennas": [
          "7",
          "8"
        ]
      },
      {
        "tunnel_no": 5,
        "name": "T5",
        "start_cell_id": "6B",
        "end_cell_id": "6D",
        "dead_end": false,
        "start_anchor_idx": 1,
        "end_anchor_idx": 0,
        "antenna_count": 2,
        "antennas": [
          "9",
          "10"
        ]
      },
      {
        "tunnel_no": 6,
        "name": "T6",
        "start_cell_id": "6D",
        "end_cell_id": "6F",
        "dead_end": false,
        "start_anchor_idx": 1,
        "end_anchor_idx": 0,
        "antenna_count": 2,
        "antennas": [
          "11",
          "12"
        ]
      },
      {
        "tunnel_no": 7,
        "name": "T7",
        "start_cell_id": "6F",
        "end_cell_id": "4F",
        "dead_end": false,
        "start_anchor_idx": 2,
        "end_anchor_idx": 3,
        "antenna_count": 2,
        "antennas": [
          "13",
          "14"
        ]
      },
      {
        "tunnel_no": 8,
        "name": "T8",
        "start_cell_id": "4F",
        "end_cell_id": "2F",
        "dead_end": false,
        "start_anchor_idx": 2,
        "end_anchor_idx": 3,
        "antenna_count": 2,
        "antennas": [
          "15",
          "16"
        ]
      }
    ],
    "antenna_combinations": {
      "1_2": "c1_c2",
      "2_1": "c2_c1",
      "3_4": "c2_c3",
      "4_3": "c3_c2",
      "5_6": "c3_c4",
      "6_5": "c4_c3",
      "7_8": "c4_c5",
      "8_7": "c5_c4",
      "9_10": "c5_c6",
      "10_9": "c6_c5",
      "11_12": "c6_c7",
      "12_11": "c7_c6",
      "13_14": "c7_c8",
      "14_13": "c8_c7",
      "15_16": "c8_c1",
      "16_15": "c1_c8",
      "4_4": "cage_3",
      "5_5": "cage_3",
      "4_5": "cage_3",
      "5_4": "cage_3",
      "4_6": "cage_3",
      "6_4": "cage_3",
      "5_3": "cage_3",
      "3_5": "cage_3",
      "6_6": "cage_4",
      "7_7": "cage_4",
      "6_7": "cage_4",
      "7_6": "cage_4",
      "6_8": "cage_4",
      "8_6": "cage_4",
      "7_5": "cage_4",
      "5_7": "cage_4",
      "8_8": "cage_5",
      "9_9": "cage_5",
      "8_9": "cage_5",
      "9_8": "cage_5",
      "8_10": "cage_5",
      "10_8": "cage_5",
      "9_7": "cage_5",
      "7_9": "cage_5",
      "2_2": "cage_2",
      "3_3": "cage_2",
      "2_3": "cage_2",
      "3_2": "cage_2",
      "2_4": "cage_2",
      "4_2": "cage_2",
      "3_1": "cage_2",
      "1_3": "cage_2",
      "10_10": "cage_6",
      "11_11": "cage_6",
      "10_11": "cage_6",
      "11_10": "cage_6",
      "10_12": "cage_6",
      "12_10": "cage_6",
      "11_9": "cage_6",
      "9_11": "cage_6",
      "1_1": "cage_1",
      "16_16": "cage_1",
      "1_16": "cage_1",
      "16_1": "cage_1",
      "1_15": "cage_1",
      "15_1": "cage_1",
      "16_2": "cage_1",
      "2_16": "cage_1",
      "14_14": "cage_8",
      "15_15": "cage_8",
      "14_15": "cage_8",
      "15_14": "cage_8",
      "14_16": "cage_8",
      "16_14": "cage_8",
      "15_13": "cage_8",
      "13_15": "cage_8",
      "12_12": "cage_7",
      "13_13": "cage_7",
      "12_13": "cage_7",
      "13_12": "cage_7",
      "12_14": "cage_7",
      "14_12": "cage_7",
      "13_11": "cage_7",
      "11_13": "cage_7"
    },
    "tunnels_map": {
      "c1_c2": "tunnel_1",
      "c2_c1": "tunnel_1",
      "c2_c3": "tunnel_2",
      "c3_c2": "tunnel_2",
      "c3_c4": "tunnel_3",
      "c4_c3": "tunnel_3",
      "c4_c5": "tunnel_4",
      "c5_c4": "tunnel_4",
      "c5_c6": "tunnel_5",
      "c6_c5": "tunnel_5",
      "c6_c7": "tunnel_6",
      "c7_c6": "tunnel_6",
      "c7_c8": "tunnel_7",
      "c8_c7": "tunnel_7",
      "c8_c1": "tunnel_8",
      "c1_c8": "tunnel_8"
    }
  }
}''')


from PySide6.QtCore import Qt, QTimer, QDate, QPointF, QRectF
from PySide6.QtGui import (
    QImage, QPixmap, QStandardItemModel, QStandardItem,
    QPen, QBrush, QColor, QPalette, QPainter, QPainterPath, QFont
)
from PySide6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QLabel, QLineEdit, QPushButton, QComboBox,
    QSpinBox, QCheckBox, QFileDialog, QMessageBox, QGroupBox, QGridLayout, QHBoxLayout,
    QVBoxLayout, QSplitter, QTableWidget, QTableWidgetItem, QScrollArea, QFormLayout,
    QSizePolicy, QRadioButton, QButtonGroup, QDateEdit, QAbstractItemView, QDialog,
    QGraphicsView, QGraphicsScene, QGraphicsRectItem, QGraphicsLineItem, QGraphicsEllipseItem,
    QGraphicsProxyWidget, QGraphicsTextItem
)

try:
    from PySide6.QtSvg import QSvgGenerator
    _HAS_SVG = True
except Exception:
    _HAS_SVG = False




# -----------------------------
# Old EcoHAB RFID board configuration
# -----------------------------
RFID_CHANNELS = [str(i) for i in range(1, 9)]
TTL_CHANNELS: List[str] = []
RFID_ACTIVE_WINDOW_S = 1.5
STATUS_UPDATE_MIN_INTERVAL_S = 0.05
TAG_MERGE_INTERVAL_S = 1.0


def sort_rfid_channels(chs: List[str]) -> List[str]:
    return sorted([str(c) for c in chs], key=lambda x: int(x) if str(x).isdigit() else 999)


RFID_CHANNELS = sort_rfid_channels(RFID_CHANNELS)


def safe_slug(s: str, fallback: str = "experiment") -> str:
    s = (s or "").strip()
    if not s:
        return fallback
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9._-]", "", s)
    return s or fallback


def sanitize_token(s: str, fallback: str = "") -> str:
    s = (s or "").strip()
    if not s:
        return fallback
    s = re.sub(r"[^\w\-]", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or fallback


def parse_old_board_line(line: bytes) -> Tuple[str, str, str]:
    """Parse old-board firmware lines: zero_based_channel-board_timestamp-rfid_hex."""
    s = line.decode("utf-8", errors="ignore").strip()
    parts = s.split("-")
    if len(parts) != 3:
        raise ValueError("Bad old-board line")
    ch0, board_t, tag = [p.strip() for p in parts]
    ch = str(int(ch0) + 1)  # firmware is 0-based; GUI/downstream use 1-based antennas
    if len(board_t) != 10:
        raise ValueError("Bad board timestamp length")
    if len(tag) != 10:
        raise ValueError("Bad RFID tag length")
    return ch, tag, board_t


def make_spinbox(vmin: int, vmax: int, value: int, step: int = 1) -> QSpinBox:
    sb = QSpinBox()
    sb.setRange(int(vmin), int(vmax))
    sb.setSingleStep(int(step))
    sb.setValue(int(value))
    sb.setAccelerated(True)
    sb.setKeyboardTracking(False)
    sb.setFocusPolicy(Qt.StrongFocus)
    sb.setButtonSymbols(QSpinBox.UpDownArrows)
    return sb


def list_serial_ports() -> List[str]:
    return [p.device for p in serial.tools.list_ports.comports()]


def apply_dark_theme(app: QApplication) -> None:
    """DeepEcoHAB dark theme."""
    try:
        app.setStyle("Fusion")
    except Exception:
        pass
    pal = QPalette()
    pal.setColor(QPalette.Window, QColor("#0f1115"))
    pal.setColor(QPalette.WindowText, QColor("#e6e6e6"))
    pal.setColor(QPalette.Base, QColor("#12151c"))
    pal.setColor(QPalette.AlternateBase, QColor("#0f1115"))
    pal.setColor(QPalette.ToolTipBase, QColor("#0f1115"))
    pal.setColor(QPalette.ToolTipText, QColor("#e6e6e6"))
    pal.setColor(QPalette.Text, QColor("#e6e6e6"))
    pal.setColor(QPalette.Button, QColor("#151a23"))
    pal.setColor(QPalette.ButtonText, QColor("#e6e6e6"))
    pal.setColor(QPalette.BrightText, QColor("#ffffff"))
    pal.setColor(QPalette.Highlight, QColor("#2d6cdf"))
    pal.setColor(QPalette.HighlightedText, QColor("#ffffff"))
    app.setPalette(pal)
    app.setStyleSheet("""
        QWidget { background: #0f1115; color: #e6e6e6; font-family: Segoe UI, Arial; font-size: 10pt; }
        QLabel, QCheckBox, QRadioButton { background: transparent; color: #e6e6e6; }
        QToolTip { color: #e6e6e6; background-color: #0f1115; border: 1px solid #2b2f3a; }
        QGroupBox { background: #151a23; border: 1px solid #2b2f3a; border-radius: 8px; margin-top: 10px; padding: 8px; font-weight: 600; }
        QGroupBox::title { subcontrol-origin: margin; left: 10px; padding: 0 4px; background: transparent; color: #e6e6e6; }
        QLineEdit, QComboBox, QSpinBox, QDateEdit { background: #12151c; color: #e6e6e6; border: 1px solid #2b2f3a; border-radius: 6px; padding: 4px 6px; min-height: 24px; }
        QPushButton { background: #1a2230; color: #e6e6e6; border: 1px solid #2b2f3a; border-radius: 8px; padding: 6px 10px; font-weight: 600; }
        QPushButton:hover { background: #22304a; border: 1px solid #2d6cdf; }
        QPushButton:pressed { background: #2d6cdf; color: #ffffff; }
        QPushButton:disabled { background: #11151d; color: #7c7c7c; }
        QTableWidget { background: #12151c; gridline-color: #2b2f3a; color: #e6e6e6; }
        QHeaderView::section { background: #151a23; border: 1px solid #2b2f3a; padding: 4px; color: #e6e6e6; }
        QScrollArea { border: none; }
    """)


def led_style(active: bool) -> str:
    color = "#2ecc71" if active else "#b0b0b0"
    border = "#1e8449" if active else "#7a7a7a"
    return (
        "min-width:12px; min-height:12px; max-width:12px; max-height:12px;"
        f"border-radius:6px; background:{color}; border:1px solid {border};"
    )


def read_one_rfid_event_from_port(port: str, baud: int = 115200, timeout_s: float = 90.0) -> Optional[Tuple[str, str]]:
    """Return (channel, tag) for first old-board RFID event observed."""
    ser = None
    try:
        ser = serial.Serial(port, baud, timeout=0.2)
        try:
            ser.reset_input_buffer()
        except Exception:
            pass
        deadline = time.time() + timeout_s
        while time.time() < deadline:
            raw = ser.readline()
            if not raw:
                continue
            try:
                ch, tag, _ = parse_old_board_line(raw)
                return ch, tag
            except Exception:
                pass
        return None
    except Exception:
        return None
    finally:
        try:
            if ser:
                ser.close()
        except Exception:
            pass

# -----------------------------
# Shared config (layout + animals) writer/reader
# -----------------------------
CSV_COLUMNS = [
    "section", "key", "value",
    "row_index", "mouse_line", "genotype", "number", "sex", "date_of_birth", "tag_no"
]


KNOWN_LAYOUT_KEYS = [
    "cages", "tunnels", "antenna_combinations_interp", "antenna_combinations", "tunnels_map"
]


def new_config_skeleton(exp_name: str) -> Dict[str, Any]:
    return {
        "version": 2,
        "experiment": {
            "name": exp_name,
            "updated": dt.datetime.now().isoformat(),
            "start_datetime": "",
            "recording_timezone": "Europe/Warsaw",
        },
        "layout": {
            "cages": {},
            "tunnels": {},
            "antenna_combinations_interp": {},
            "antenna_combinations": {},
            "tunnels_map": {},
        },
        "animals": {
            "n_mice": 0,
            "rows": {}
        },
        "environment": {},
        "hardware_trigger": {},
        "camera_trigger": {}
    }


def _as_int_or_none(value: Any) -> Optional[int]:
    try:
        if value is None or value == "":
            return None
        return int(value)
    except Exception:
        return None


def _layout_cage_records(layout: Dict[str, Any]) -> List[Tuple[str, Dict[str, Any]]]:
    """Return cages as [(cage_key, cage_record), ...], accepting old list and new dict formats."""
    cages = (layout or {}).get("cages") or {}
    out: List[Tuple[str, Dict[str, Any]]] = []
    if isinstance(cages, dict):
        for key, rec in cages.items():
            if not isinstance(rec, dict):
                continue
            rec2 = dict(rec)
            m = re.search(r"(\d+)", str(key))
            if "cage_no" not in rec2 and m:
                rec2["cage_no"] = m.group(1)
            out.append((str(key), rec2))
    elif isinstance(cages, list):
        for idx, rec in enumerate(cages, start=1):
            if not isinstance(rec, dict):
                continue
            rec2 = dict(rec)
            cage_no = str(rec2.get("cage_no") or idx)
            out.append((f"cage_{cage_no}", rec2))
    return out


def _layout_tunnel_records(layout: Dict[str, Any]) -> List[Tuple[str, Dict[str, Any]]]:
    """Return tunnels as [(tunnel_key, tunnel_record), ...], accepting old list and new dict formats."""
    tunnels = (layout or {}).get("tunnels") or {}
    out: List[Tuple[str, Dict[str, Any]]] = []
    if isinstance(tunnels, dict):
        for key, rec in tunnels.items():
            if not isinstance(rec, dict):
                continue
            rec2 = dict(rec)
            m = re.search(r"(\d+)", str(key))
            if "tunnel_no" not in rec2 and m:
                rec2["tunnel_no"] = int(m.group(1))
            out.append((str(key), rec2))
    elif isinstance(tunnels, list):
        for idx, rec in enumerate(tunnels, start=1):
            if not isinstance(rec, dict):
                continue
            rec2 = dict(rec)
            tunnel_no = _as_int_or_none(rec2.get("tunnel_no")) or idx
            out.append((f"tunnel_{tunnel_no}", rec2))
    return out


def _derive_compact_antenna_combinations(layout: Dict[str, Any]) -> Dict[str, str]:
    """Build the downstream compact table: direct tunnel pairs + within-cage prox/prox pairs.

    The richer interpreted table keeps all generalized within-cage possibilities; this compact table
    matches the structure used by the downstream pipeline example config.
    """
    cell_to_cage_no: Dict[str, str] = {}
    for cage_key, rec in _layout_cage_records(layout):
        cid = str(rec.get("cell_id") or "").strip()
        if not cid:
            continue
        no = str(rec.get("cage_no") or re.sub(r"\D+", "", cage_key) or "").strip()
        if no:
            cell_to_cage_no[cid] = no

    compact: Dict[str, str] = {}
    cage_inc: Dict[str, List[Tuple[str, str]]] = {cid: [] for cid in cell_to_cage_no.keys()}

    for _tkey, rec in _layout_tunnel_records(layout):
        if rec.get("dead_end"):
            continue
        sc = str(rec.get("start_cell_id") or "").strip()
        ec = str(rec.get("end_cell_id") or "").strip()
        if sc not in cell_to_cage_no or ec not in cell_to_cage_no:
            continue
        ants = rec.get("antennas") or []
        if not isinstance(ants, list) or len(ants) < 2:
            continue
        a_start = str(ants[0] or "").strip()
        a_end = str(ants[-1] or "").strip()
        if not a_start or a_start == "?" or not a_end or a_end == "?":
            continue

        cA = cell_to_cage_no[sc]
        cB = cell_to_cage_no[ec]
        compact[f"{a_start}_{a_end}"] = f"c{cA}_c{cB}"
        compact[f"{a_end}_{a_start}"] = f"c{cB}_c{cA}"

        cage_inc.setdefault(sc, []).append((a_start, a_end))
        cage_inc.setdefault(ec, []).append((a_end, a_start))

    for cid, pairs in cage_inc.items():
        cage_no = cell_to_cage_no.get(cid, "")
        if not cage_no:
            continue
        cage_label = f"cage_{cage_no}"
        prox_list = [p for p, _opp in pairs if p and p != "?"]
        for prox in prox_list:
            compact[f"{prox}_{prox}"] = cage_label
        for i in range(len(prox_list)):
            for j in range(i + 1, len(prox_list)):
                p_i, p_j = prox_list[i], prox_list[j]
                compact[f"{p_i}_{p_j}"] = cage_label
                compact[f"{p_j}_{p_i}"] = cage_label
    return compact


def normalize_layout_for_config(layout: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """Convert old v10 layout lists into downstream-style dictionaries while preserving extra fields."""
    layout = dict(layout or {})

    cage_dict: Dict[str, Dict[str, Any]] = {}
    for cage_key, rec in _layout_cage_records(layout):
        rec2 = dict(rec)
        cage_no = str(rec2.get("cage_no") or re.sub(r"\D+", "", cage_key) or "").strip()
        out_key = f"cage_{cage_no}" if cage_no else str(cage_key)
        # Keep cage_no as extra metadata for backwards compatibility with GUI exports.
        if cage_no and "cage_no" not in rec2:
            rec2["cage_no"] = cage_no
        cage_dict[out_key] = rec2

    tunnel_dict: Dict[str, Dict[str, Any]] = {}
    for tunnel_key, rec in _layout_tunnel_records(layout):
        rec2 = dict(rec)
        tunnel_no = _as_int_or_none(rec2.get("tunnel_no")) or _as_int_or_none(re.sub(r"\D+", "", tunnel_key))
        out_key = f"tunnel_{tunnel_no}" if tunnel_no else str(tunnel_key)
        if tunnel_no is not None:
            rec2["tunnel_no"] = int(tunnel_no)
        tunnel_dict[out_key] = rec2

    # v10 generated the rich/interpreted combination table as antenna_combinations.
    # Preserve it as antenna_combinations_interp, and generate a compact downstream table separately.
    rich_combos = layout.get("antenna_combinations_interp")
    if not isinstance(rich_combos, dict) or not rich_combos:
        rich_combos = layout.get("antenna_combinations") if isinstance(layout.get("antenna_combinations"), dict) else {}

    compact_combos = _derive_compact_antenna_combinations({**layout, "cages": cage_dict, "tunnels": tunnel_dict})
    if not compact_combos:
        compact_combos = layout.get("antenna_combinations") if isinstance(layout.get("antenna_combinations"), dict) else {}

    normalized = dict(layout)
    normalized["cages"] = cage_dict
    normalized["tunnels"] = tunnel_dict
    normalized["antenna_combinations_interp"] = dict(rich_combos or {})
    normalized["antenna_combinations"] = dict(compact_combos or {})
    normalized["tunnels_map"] = dict(layout.get("tunnels_map") or {})
    return normalized


def _animal_rows_as_list(animals: Dict[str, Any]) -> List[Dict[str, Any]]:
    rows = (animals or {}).get("rows") or []
    out: List[Dict[str, Any]] = []
    if isinstance(rows, dict):
        for tag_key, rec in rows.items():
            if not isinstance(rec, dict):
                continue
            rec2 = dict(rec)
            if "tag_no" not in rec2 and not str(tag_key).startswith("row_"):
                rec2["tag_no"] = str(tag_key)
            out.append(rec2)
    elif isinstance(rows, list):
        for rec in rows:
            if isinstance(rec, dict):
                out.append(dict(rec))
    out.sort(key=lambda r: (_as_int_or_none(r.get("row_index")) or 10**9, str(r.get("tag_no") or "")))
    return out


def normalize_animals_for_config(animals: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    animals = dict(animals or {})
    row_dict: Dict[str, Dict[str, Any]] = {}
    rows = _animal_rows_as_list(animals)
    for i, rec in enumerate(rows, start=1):
        rec2 = dict(rec)
        rec2["row_index"] = _as_int_or_none(rec2.get("row_index")) or i
        tag = str(rec2.get("tag_no") or "").strip()
        key = tag if tag else f"row_{rec2['row_index']}"
        # Preserve tag_no as extra metadata; downstream can still use the dict key as in the example.
        rec2["tag_no"] = tag
        row_dict[key] = rec2
    animals["n_mice"] = int(animals.get("n_mice") or len(row_dict) or 0)
    animals["rows"] = row_dict
    return animals


def _csv_value(v: Any) -> str:
    if isinstance(v, (dict, list, bool)) or v is None:
        return json.dumps(v, ensure_ascii=False)
    return str(v)


def _append_kv(rows: List[Dict[str, Any]], section: str, key: str, value: Any) -> None:
    rows.append({
        "section": section,
        "key": str(key),
        "value": _csv_value(value),
        "row_index": "", "mouse_line": "", "genotype": "", "number": "", "sex": "", "date_of_birth": "", "tag_no": ""
    })


def write_config_csv(cfg: Dict[str, Any], csv_path: Path) -> None:
    """Write a flat CSV mirror of the v11 JSON hierarchy.

    The columns are kept compatible with earlier exports, but keys now use stable hierarchical
    names such as layout.cages.cage_1.cell_id and animals.<RFID_TAG>.
    """
    rows: List[Dict[str, Any]] = []
    cfg = dict(cfg or {})
    layout = normalize_layout_for_config(cfg.get("layout") or {})
    animals = normalize_animals_for_config(cfg.get("animals") or {})

    exp = cfg.get("experiment", {}) or {}
    for k, v in exp.items():
        _append_kv(rows, "experiment", k, v)

    for cage_key, rec in (layout.get("cages") or {}).items():
        for k, v in rec.items():
            _append_kv(rows, "layout.cages", f"{cage_key}.{k}", v)

    for tunnel_key, rec in (layout.get("tunnels") or {}).items():
        for k, v in rec.items():
            _append_kv(rows, "layout.tunnels", f"{tunnel_key}.{k}", v)

    for section_name in ["antenna_combinations_interp", "antenna_combinations", "tunnels_map"]:
        section_obj = layout.get(section_name) or {}
        for k, v in sorted(section_obj.items(), key=lambda kv: str(kv[0])):
            _append_kv(rows, f"layout.{section_name}", str(k), v)

    _append_kv(rows, "animals", "n_mice", animals.get("n_mice", 0))
    for tag_key, r in (animals.get("rows") or {}).items():
        rows.append({
            "section": "animals.rows",
            "key": str(tag_key),
            "value": "",
            "row_index": r.get("row_index", ""),
            "mouse_line": r.get("mouse_line", ""),
            "genotype": r.get("genotype", ""),
            "number": r.get("number", ""),
            "sex": r.get("sex", ""),
            "date_of_birth": r.get("date_of_birth", ""),
            "tag_no": r.get("tag_no", str(tag_key) if not str(tag_key).startswith("row_") else ""),
        })

    for block_name in ["environment", "hardware_trigger"]:
        block = cfg.get(block_name, {}) or {}
        for k, v in sorted(block.items(), key=lambda kv: str(kv[0])):
            _append_kv(rows, block_name, str(k), v)

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in CSV_COLUMNS})

def read_config_json(path: Path) -> Optional[Dict[str, Any]]:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def write_config_json(cfg: Dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)




# -----------------------------
# Old board acquisition
# -----------------------------
class OldBoardAcquisition:
    def __init__(self, port: str, save_dir: str, file_prefix: str, baudrate: int = 115200):
        self.port = port
        self.save_dir = str(save_dir)
        self.file_prefix = file_prefix
        self.baudrate = int(baudrate)
        Path(self.save_dir).mkdir(parents=True, exist_ok=True)
        self.data_file = Path(self.save_dir) / f"{file_prefix}_old_board_rfid_full_recording.csv"
        self.raw_serial_file = Path(self.save_dir) / f"{file_prefix}_old_board_raw_serial.log"
        self.unparsed_file = Path(self.save_dir) / f"{file_prefix}_old_board_unparsed_lines.log"

        self._stop = threading.Event()
        self._thread: Optional[threading.Thread] = None
        self._ser: Optional[serial.Serial] = None
        self._lock = threading.Lock()

        self.lines_ok = 0
        self.lines_bad = 0
        self.bytes_read = 0
        self.last_by_channel: Dict[str, float] = {}
        self.last_tag_by_channel: Dict[str, str] = {}
        self.event_counter = 0
        self.last_event: Tuple[int, str, str] = (0, "", "")
        self.tag_counts: Dict[str, int] = {}
        self.last_raw_line = ""
        self.last_bad_line = ""
        self._running = False

    def start(self):
        if self._thread and self._thread.is_alive():
            return
        self._stop.clear()
        self._thread = threading.Thread(target=self._reader_loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop.set()
        try:
            if self._ser:
                self._ser.close()
        except Exception:
            pass

    def is_alive(self) -> bool:
        return self._thread is not None and self._thread.is_alive()

    def _note_event(self, ch: str, tag: str):
        now = time.monotonic()
        ch = str(ch)
        tag = str(tag)
        with self._lock:
            self.last_by_channel[ch] = now
            self.last_tag_by_channel[ch] = tag
            self.event_counter += 1
            self.last_event = (self.event_counter, tag, ch)
            self.tag_counts[tag] = self.tag_counts.get(tag, 0) + 1

    def get_status_snapshot(self) -> Tuple[Dict[str, bool], Dict[str, int], Dict[str, str]]:
        now = time.monotonic()
        with self._lock:
            active = {ch: (now - self.last_by_channel.get(ch, -1e9) <= RFID_ACTIVE_WINDOW_S) for ch in RFID_CHANNELS}
            counts = dict(self.tag_counts)
            last_tags = dict(self.last_tag_by_channel)
        return active, counts, last_tags

    def get_unique_tags_snapshot(self) -> Dict[str, int]:
        with self._lock:
            return dict(self.tag_counts)

    def is_running(self) -> bool:
        return self.is_alive()

    def get_last_rfid_event(self) -> Tuple[int, str, str]:
        with self._lock:
            return self.last_event

    def _reader_loop(self):
        raw_f = None
        bad_f = None
        try:
            self._ser = serial.Serial(self.port, self.baudrate, timeout=0.2)
            self._running = True
            raw_f = open(self.raw_serial_file, "a", encoding="utf-8", buffering=1)
            bad_f = open(self.unparsed_file, "a", encoding="utf-8", buffering=1)
            raw_f.write(f"----- Old board raw serial start {dt.datetime.now().isoformat()} -----\n")
            bad_f.write(f"----- Old board unparsed start {dt.datetime.now().isoformat()} -----\n")

            new_file = not self.data_file.exists() or self.data_file.stat().st_size == 0
            with open(self.data_file, "a", newline="", encoding="utf-8") as csvfile:
                writer = csv.writer(csvfile)
                if new_file:
                    writer.writerow(["PC Time", "Channel", "Board Time", "RFID Tag"])
                while not self._stop.is_set():
                    raw = self._ser.readline()
                    if not raw:
                        continue
                    self.bytes_read += len(raw)
                    raw_text = raw.decode("utf-8", errors="replace").strip()
                    self.last_raw_line = raw_text
                    try:
                        raw_f.write(f"{dt.datetime.now().isoformat()};{raw_text}\n")
                    except Exception:
                        pass
                    try:
                        ch, tag, board_t = parse_old_board_line(raw)
                        pc_time = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S:%f")[:-3]
                        writer.writerow([pc_time, ch, board_t, tag])
                        csvfile.flush()
                        self.lines_ok += 1
                        self._note_event(ch, tag)
                    except Exception as e:
                        self.lines_bad += 1
                        self.last_bad_line = raw_text
                        try:
                            bad_f.write(f"{dt.datetime.now().isoformat()};{raw_text};{e}\n")
                        except Exception:
                            pass
        except Exception as e:
            self.last_bad_line = f"Serial/open error: {e}"
        finally:
            self._running = False
            for fh in (raw_f, bad_f):
                try:
                    if fh:
                        fh.write(f"----- Old board serial end {dt.datetime.now().isoformat()} -----\n")
                        fh.flush(); fh.close()
                except Exception:
                    pass
            try:
                if self._ser:
                    self._ser.close()
            except Exception:
                pass

class AnimalDetailsDialog(QDialog):
    MATCH_TIMEOUT_S = 90

    def __init__(self, main: "MainWindow"):
        super().__init__(main)
        self.main = main
        self.setWindowTitle("Animal details")
        self.resize(1120, 720)
        self._sync_guard = False
        self._same_guard = False

        self.waiting_row: Optional[int] = None
        self.waiting_remaining: int = 0
        self.waiting_start_counter: Optional[int] = None
        self._direct_thread: Optional[threading.Thread] = None
        self._direct_result: Dict[str, Any] = {"done": False, "tag": None}
        self._direct_deadline: float = 0.0

        self._mouse_widgets: List[Dict[str, Any]] = []

        self._build_ui()
        self._connect_sync()
        self._rebuild_rows(self.n_mice_spin.value())

        self.poll_timer = QTimer(self)
        self.poll_timer.setInterval(200)
        self.poll_timer.timeout.connect(self._poll_match)

        self.countdown_timer = QTimer(self)
        self.countdown_timer.setInterval(1000)
        self.countdown_timer.timeout.connect(self._tick_countdown)

    def _build_ui(self):
        root = QWidget()
        lay = QVBoxLayout(root)
        lay.setContentsMargins(10, 10, 10, 10)
        lay.setSpacing(10)
        self.setLayout(lay)

        top = QHBoxLayout()
        top.addWidget(QLabel("Experiment name:"))
        self.exp_name_edit = QLineEdit(self.main.exp_edit.text())
        self.exp_name_edit.setMaximumWidth(340)
        top.addWidget(self.exp_name_edit)

        self.btn_import = QPushButton("Import JSON")
        self.btn_import.clicked.connect(self._import_json)
        top.addWidget(self.btn_import)

        self.btn_export_both = QPushButton("Export to CSV & JSON")
        self.btn_export_both.clicked.connect(lambda: self._export_details(fmt="csv"))
        top.addWidget(self.btn_export_both)

        top.addStretch(1)
        lay.addLayout(top)

        nm = QHBoxLayout()
        nm.addWidget(QLabel("Number of mice in the experiment:"))
        self.n_mice_spin = make_spinbox(1, 50, 12)
        self.n_mice_spin.valueChanged.connect(lambda v: self._rebuild_rows(int(v)))
        nm.addWidget(self.n_mice_spin)
        nm.addStretch(1)
        lay.addLayout(nm)

        # Same-for-all row
        same_box = QGroupBox("Same for all")
        sg = QGridLayout(same_box)
        sg.setContentsMargins(8, 8, 8, 8)
        sg.setHorizontalSpacing(10)
        sg.setVerticalSpacing(4)

        self.same_line = QCheckBox("Mouse line")
        self.same_geno = QCheckBox("Genotype")
        self.same_sex = QCheckBox("Sex")
        self.same_dob = QCheckBox("Date of birth")
        sg.addWidget(self.same_line, 0, 0)
        sg.addWidget(self.same_geno, 0, 1)
        sg.addWidget(self.same_sex, 0, 2)
        sg.addWidget(self.same_dob, 0, 3)
        sg.setColumnStretch(4, 1)

        self.same_line.toggled.connect(lambda _: self._apply_same_all("line"))
        self.same_geno.toggled.connect(lambda _: self._apply_same_all("geno"))
        self.same_sex.toggled.connect(lambda _: self._apply_same_all("sex"))
        self.same_dob.toggled.connect(lambda _: self._apply_same_all("dob"))

        lay.addWidget(same_box)

        self.status_lbl = QLabel("Ready.")
        lay.addWidget(self.status_lbl)

        self.table = QTableWidget(0, 8)
        self.table.setHorizontalHeaderLabels([
            "No.", "Mouse line", "Genotype", "Number", "Sex", "Date of birth", "Tag no.", "Match tag"
        ])
        self.table.verticalHeader().setVisible(False)
        self.table.setSortingEnabled(False)
        self.table.setEditTriggers(QTableWidget.NoEditTriggers)
        self.table.setSelectionMode(QAbstractItemView.NoSelection)

        self.table.setColumnWidth(0, 50)
        self.table.setColumnWidth(1, 170)
        self.table.setColumnWidth(2, 170)
        self.table.setColumnWidth(3, 90)
        self.table.setColumnWidth(4, 85)
        self.table.setColumnWidth(5, 140)
        self.table.setColumnWidth(6, 220)
        self.table.setColumnWidth(7, 130)

        lay.addWidget(self.table, 1)

        hint = QLabel(
            "Match tag: press the row's button, then pass the animal through any RFID antenna. "
            "The software waits 90 seconds and shows a countdown."
        )
        hint.setStyleSheet("color:#555;")
        lay.addWidget(hint)

    def _connect_sync(self):
        def from_main(txt: str):
            if self._sync_guard: return
            self._sync_guard = True
            try:
                if self.exp_name_edit.text() != txt:
                    self.exp_name_edit.setText(txt)
            finally:
                self._sync_guard = False

        def from_dialog(txt: str):
            if self._sync_guard: return
            self._sync_guard = True
            try:
                if self.main.exp_edit.text() != txt:
                    self.main.exp_edit.setText(txt)
            finally:
                self._sync_guard = False

        self.main.exp_edit.textChanged.connect(from_main)
        self.exp_name_edit.textChanged.connect(from_dialog)

    def _gather_row_data(self) -> List[Dict[str, Any]]:
        out = []
        for w in self._mouse_widgets:
            out.append({
                "mouse_line": w["line"].text(),
                "genotype": w["geno"].text(),
                "number": w["number"].text(),
                "sex": w["sex"].currentText(),
                "dob": w["dob"].date().toString("yyyy-MM-dd"),
                "tag": w["tag"].text(),
            })
        return out

    def _disconnect_first_row_signals(self):
        # Best-effort safe disconnect (Qt raises if not connected)
        if not self._mouse_widgets:
            return
        r0 = self._mouse_widgets[0]
        for sig, fn in [
            (r0["line"].textChanged, self._same_line_from_first),
            (r0["geno"].textChanged, self._same_geno_from_first),
            (r0["sex"].currentTextChanged, self._same_sex_from_first),
            (r0["dob"].dateChanged, self._same_dob_from_first),
        ]:
            try:
                sig.disconnect(fn)
            except Exception:
                pass

    def _connect_first_row_signals(self):
        if not self._mouse_widgets:
            return
        r0 = self._mouse_widgets[0]
        r0["line"].textChanged.connect(self._same_line_from_first)
        r0["geno"].textChanged.connect(self._same_geno_from_first)
        r0["sex"].currentTextChanged.connect(self._same_sex_from_first)
        r0["dob"].dateChanged.connect(self._same_dob_from_first)

    def _rebuild_rows(self, n: int):
        n = max(1, min(50, int(n)))
        old = self._gather_row_data()

        self._disconnect_first_row_signals()

        self.table.setRowCount(n)
        self._mouse_widgets = []

        for r in range(n):
            row: Dict[str, Any] = {}

            item = QTableWidgetItem(str(r + 1))
            item.setTextAlignment(Qt.AlignCenter)
            self.table.setItem(r, 0, item)

            line = QLineEdit()
            self.table.setCellWidget(r, 1, line)
            row["line"] = line

            geno = QLineEdit()
            self.table.setCellWidget(r, 2, geno)
            row["geno"] = geno

            number = QLineEdit()
            self.table.setCellWidget(r, 3, number)
            row["number"] = number

            sex = QComboBox()
            sex.addItems(["Male", "Female"])
            self.table.setCellWidget(r, 4, sex)
            row["sex"] = sex

            dob = QDateEdit()
            dob.setCalendarPopup(True)
            dob.setDisplayFormat("yyyy-MM-dd")
            dob.setDate(QDate.currentDate())
            self.table.setCellWidget(r, 5, dob)
            row["dob"] = dob

            tag = QLineEdit()
            tag.setMaxLength(64)
            self.table.setCellWidget(r, 6, tag)
            row["tag"] = tag

            btn = QPushButton("Match tag")
            btn.clicked.connect(lambda _=False, rr=r: self._on_match_tag(rr))
            self.table.setCellWidget(r, 7, btn)
            row["match"] = btn

            self._mouse_widgets.append(row)

        for r in range(min(len(old), n)):
            self._mouse_widgets[r]["line"].setText(old[r].get("mouse_line", ""))
            self._mouse_widgets[r]["geno"].setText(old[r].get("genotype", ""))
            self._mouse_widgets[r]["number"].setText(old[r].get("number", ""))
            sex_txt = old[r].get("sex", "Male")
            self._mouse_widgets[r]["sex"].setCurrentText(sex_txt if sex_txt in ("Male", "Female") else "Male")
            dob_txt = old[r].get("dob", "")
            if dob_txt:
                try:
                    y, m, d = [int(x) for x in dob_txt.split("-")]
                    self._mouse_widgets[r]["dob"].setDate(QDate(y, m, d))
                except Exception:
                    pass
            self._mouse_widgets[r]["tag"].setText(old[r].get("tag", ""))

        self._connect_first_row_signals()
        self._apply_same_all("line")
        self._apply_same_all("geno")
        self._apply_same_all("sex")
        self._apply_same_all("dob")

        self.status_lbl.setText(f"Ready. Rows: {n}")

    # ---- Same-for-all propagation ----
    def _same_line_from_first(self, txt: str):
        if self.same_line.isChecked():
            self._apply_same_all("line")

    def _same_geno_from_first(self, txt: str):
        if self.same_geno.isChecked():
            self._apply_same_all("geno")

    def _same_sex_from_first(self, txt: str):
        if self.same_sex.isChecked():
            self._apply_same_all("sex")

    def _same_dob_from_first(self, _d: QDate):
        if self.same_dob.isChecked():
            self._apply_same_all("dob")

    def _apply_same_all(self, field: str):
        if self._same_guard:
            return
        if not self._mouse_widgets:
            return
        self._same_guard = True
        try:
            r0 = self._mouse_widgets[0]
            if field == "line" and self.same_line.isChecked():
                val = r0["line"].text()
                for r in self._mouse_widgets[1:]:
                    if r["line"].text() != val:
                        r["line"].setText(val)
            if field == "geno" and self.same_geno.isChecked():
                val = r0["geno"].text()
                for r in self._mouse_widgets[1:]:
                    if r["geno"].text() != val:
                        r["geno"].setText(val)
            if field == "sex" and self.same_sex.isChecked():
                val = r0["sex"].currentText()
                for r in self._mouse_widgets[1:]:
                    if r["sex"].currentText() != val:
                        r["sex"].setCurrentText(val)
            if field == "dob" and self.same_dob.isChecked():
                val = r0["dob"].date()
                for r in self._mouse_widgets[1:]:
                    if r["dob"].date() != val:
                        r["dob"].setDate(val)
        finally:
            self._same_guard = False

    # ---- Match tag ----
    def _set_waiting_text(self, row: int, remaining: int):
        self._mouse_widgets[row]["match"].setText(f"Waiting: {remaining}")

    def _reset_btn(self, row: int):
        self._mouse_widgets[row]["match"].setText("Match tag")

    def _cancel_wait(self, reason: str):
        if self.waiting_row is None:
            return
        row = self.waiting_row
        self.waiting_row = None
        self.waiting_remaining = 0
        self.waiting_start_counter = None
        self._direct_thread = None
        self._direct_result = {"done": False, "tag": None}
        self._direct_deadline = 0.0
        self.poll_timer.stop()
        self.countdown_timer.stop()
        self._reset_btn(row)
        self.status_lbl.setText(reason)

    def _on_match_tag(self, row: int):
        if self.waiting_row is not None:
            if self.waiting_row == row:
                self._cancel_wait("Match tag: cancelled.")
                return
            QMessageBox.information(self, "Already waiting", "Already waiting on another row.")
            return

        daq_port = self.main.daq_combo.currentText().strip()
        if not daq_port:
            QMessageBox.critical(self, "Missing DAQ COM", "Select the DAQ COM port in the main window.")
            return

        self.waiting_row = row
        self.waiting_remaining = self.MATCH_TIMEOUT_S
        self._set_waiting_text(row, self.waiting_remaining)

        if self.main.board and self.main.board.is_running():
            start_counter, _, _ = self.main.board.get_last_rfid_event()
            self.waiting_start_counter = start_counter
            self.status_lbl.setText(f"Waiting for next RFID tag (row {row+1})... (DAQ running)")
            self.poll_timer.start()
            self.countdown_timer.start()
            return

        self._direct_result = {"done": False, "tag": None}
        self._direct_deadline = time.time() + self.MATCH_TIMEOUT_S

        def _worker():
            evt = read_one_rfid_event_from_port(daq_port, timeout_s=self.MATCH_TIMEOUT_S)
            self._direct_result["tag"] = (evt[1] if evt else None)
            self._direct_result["done"] = True

        self._direct_thread = threading.Thread(target=_worker, daemon=True)
        self._direct_thread.start()
        self.status_lbl.setText(f"Waiting for RFID tag (row {row+1})... (direct read)")
        self.poll_timer.start()
        self.countdown_timer.start()

    def _tick_countdown(self):
        if self.waiting_row is None:
            self.countdown_timer.stop()
            return
        self.waiting_remaining = max(0, self.waiting_remaining - 1)
        self._set_waiting_text(self.waiting_row, self.waiting_remaining)
        if self.waiting_remaining <= 0:
            self._cancel_wait("Match tag: timeout (90s).")

    def _poll_match(self):
        if self.waiting_row is None:
            self.poll_timer.stop()
            return

        if self.main.board and self.main.board.is_running() and self.waiting_start_counter is not None:
            cur_counter, tag, ch = self.main.board.get_last_rfid_event()
            if cur_counter > self.waiting_start_counter and tag:
                row = self.waiting_row
                self._mouse_widgets[row]["tag"].setText(tag)
                self._cancel_wait(f"Captured tag {tag} (antenna {ch}) for row {row+1}.")
            return

        if self._direct_thread is not None:
            if self._direct_result.get("done", False):
                row = self.waiting_row
                tag = self._direct_result.get("tag", None)
                if tag:
                    self._mouse_widgets[row]["tag"].setText(tag)
                    self._cancel_wait(f"Captured tag {tag} for row {row+1}.")
                else:
                    self._cancel_wait("Match tag: no tag detected (timeout).")
            else:
                if time.time() > self._direct_deadline + 5:
                    self._cancel_wait("Match tag: timeout (thread).")

    # ---- Import / Export shared config ----
    def _animals_payload(self) -> Dict[str, Any]:
        rows: Dict[str, Dict[str, Any]] = {}
        for i, w in enumerate(self._mouse_widgets, start=1):
            tag = w["tag"].text().strip()
            key = tag if tag else f"row_{i}"
            rows[key] = {
                "row_index": i,
                "mouse_line": w["line"].text().strip(),
                "genotype": w["geno"].text().strip(),
                "number": w["number"].text().strip(),
                "sex": w["sex"].currentText(),
                "date_of_birth": w["dob"].date().toString("yyyy-MM-dd"),
                "tag_no": tag,
            }
        return {"n_mice": int(self.n_mice_spin.value()), "rows": rows}

    def _export_details(self, fmt: str):
        try:
            self.main.update_shared_config(animals=self._animals_payload(), layout=None)
            csv_path, json_path = self.main.get_shared_config_paths()
            if fmt.lower() == "json":
                QMessageBox.information(self, "Exported", f"Updated JSON:\n{json_path}")
            else:
                QMessageBox.information(self, "Exported", f"Updated CSV:\n{csv_path}\n(and JSON:\n{json_path})")
        except Exception as e:
            QMessageBox.critical(self, "Export failed", str(e))

    def _import_json(self):
        path, _ = QFileDialog.getOpenFileName(self, "Import config JSON", self.main.save_edit.text(), "JSON files (*.json)")
        if not path:
            return
        cfg = self.main.load_shared_config_from_file(Path(path))
        if not cfg:
            QMessageBox.critical(self, "Import failed", "Could not read JSON.")
            return

        # Do not overwrite the current experiment name or output directory when loading a template.
        animals = cfg.get("animals") or {}
        rows = _animal_rows_as_list(animals)
        n = int(animals.get("n_mice") or 0) or len(rows) or 12
        n = max(1, min(50, n))
        self.n_mice_spin.setValue(n)

        # rebuild already happened by setValue, now fill
        for r in range(min(n, len(rows))):
            rr = rows[r]
            self._mouse_widgets[r]["line"].setText(rr.get("mouse_line", ""))
            self._mouse_widgets[r]["geno"].setText(rr.get("genotype", ""))
            self._mouse_widgets[r]["number"].setText(rr.get("number", ""))
            sx = rr.get("sex", "Male")
            self._mouse_widgets[r]["sex"].setCurrentText(sx if sx in ("Male", "Female") else "Male")
            dob = rr.get("date_of_birth", "")
            if dob:
                try:
                    y, m, d = [int(x) for x in dob.split("-")]
                    self._mouse_widgets[r]["dob"].setDate(QDate(y, m, d))
                except Exception:
                    pass
            self._mouse_widgets[r]["tag"].setText(rr.get("tag_no", ""))

        self.status_lbl.setText(f"Imported animals from: {Path(path).name}")

    def closeEvent(self, event):
        self._cancel_wait("Closed.")
        super().closeEvent(event)


# -----------------------------
# Layout Editor (QGraphicsView) + shared config export/import
# -----------------------------
class LayoutDialog(QDialog):
    GRID_N = 7
    CELL = 155
    MARGIN = 14
    CHECK_TIMEOUT_S = 90
    DEFAULT_ANTENNAS_PER_TUNNEL = 2

    def __init__(self, main: "MainWindow"):
        super().__init__(main)
        self.main = main
        self.setWindowTitle("DeepEcoHAB layout")
        self.resize(1180, 900)

        self.scene = QGraphicsScene(self)
        self.view = QGraphicsView(self.scene)
        self.view.setRenderHints(QPainter.Antialiasing | QPainter.TextAntialiasing)
        self.view.setDragMode(QGraphicsView.NoDrag)
        self.view.setBackgroundBrush(QBrush(QColor("#0f1115")))

        self.mode = "cage"  # cage | tunnel | select

        self.cells: Dict[Tuple[int, int], LayoutDialog.CellItem] = {}
        self.tunnels: List[LayoutDialog.TunnelItem] = []
        self.selected_tunnel: Optional[LayoutDialog.TunnelItem] = None

        self._tunnel_start: Optional[Tuple[LayoutDialog.CellItem, QPointF]] = None
        self._active_editor_dot: Optional[LayoutDialog.AntennaDot] = None

        self.active_dot: Optional[LayoutDialog.AntennaDot] = None
        self.check_remaining: int = 0
        self.check_start_counter: Optional[int] = None
        self._direct_thread: Optional[threading.Thread] = None
        self._direct_result: Dict[str, Any] = {"done": False, "ch": None}
        self._direct_deadline: float = 0.0

        self._tunnel_serial = 1

        # fonts (bigger)
        self._font_antenna = QFont()
        self._font_antenna.setPointSize(11)  # smaller so labels don't overlap
        self._font_antenna.setBold(True)

        self._build_ui()
        self._build_grid()
        self._apply_default_layout()

        self.poll_timer = QTimer(self)
        self.poll_timer.setInterval(200)
        self.poll_timer.timeout.connect(self._poll_check)

        self.countdown_timer = QTimer(self)
        self.countdown_timer.setInterval(1000)
        self.countdown_timer.timeout.connect(self._tick_check)

    # -----------------------------
    @staticmethod
    def cell_id(r: int, c: int) -> str:
        return f"{c+1}{chr(ord('A')+r)}"

    def _rc_from_id(self, s: str) -> Tuple[int, int]:
        m = re.match(r"^([1-7])([A-G])$", s.strip().upper())
        if not m:
            raise ValueError(f"Bad cell id: {s}")
        c = int(m.group(1)) - 1
        r = ord(m.group(2)) - ord("A")
        return r, c

    # -----------------------------
    def _build_ui(self):
        root = QWidget()
        lay = QVBoxLayout(root)
        lay.setContentsMargins(10, 10, 10, 10)
        lay.setSpacing(10)
        self.setLayout(lay)

        top = QHBoxLayout()
        self.btn_cage = QPushButton("Cage tool")
        self.btn_tunnel = QPushButton("Tunnel tool")
        self.btn_select = QPushButton("Select tool")
        self.btn_cage.clicked.connect(lambda: self._set_mode("cage"))
        self.btn_tunnel.clicked.connect(lambda: self._set_mode("tunnel"))
        self.btn_select.clicked.connect(lambda: self._set_mode("select"))
        top.addWidget(self.btn_cage)
        top.addWidget(self.btn_tunnel)
        top.addWidget(self.btn_select)

        top.addSpacing(14)
        top.addWidget(QLabel("Layout preset:"))
        self.preset_combo = QComboBox()
        self.preset_combo.setMinimumWidth(210)
        self.preset_combo.blockSignals(True)
        self.preset_combo.addItems(["Custom (empty)", "Classic DeepEcoHAB", "Field DeepEcoHAB"])
        self.preset_combo.setCurrentIndex(0)
        self.preset_combo.blockSignals(False)
        self.preset_combo.currentIndexChanged.connect(self._on_preset_selected)
        top.addWidget(self.preset_combo)

        top.addSpacing(14)
        top.addWidget(QLabel("Antennas on selected tunnel:"))
        self.ant_spin = make_spinbox(0, 20, self.DEFAULT_ANTENNAS_PER_TUNNEL)
        self.ant_spin.valueChanged.connect(self._apply_antenna_count)
        top.addWidget(self.ant_spin)

        self.btn_delete = QPushButton("Delete selected tunnel")
        self.btn_delete.clicked.connect(self._delete_selected)
        top.addWidget(self.btn_delete)

        top.addSpacing(16)
        top.addWidget(QLabel("Tunnel name:"))
        self.tname_edit = QLineEdit("")
        self.tname_edit.setFixedWidth(220)
        self.tname_edit.textEdited.connect(self._on_tunnel_name_changed)
        top.addWidget(self.tname_edit)

        top.addStretch(1)

        self.btn_import = QPushButton("Import JSON")
        self.btn_import.clicked.connect(self._import_json)
        top.addWidget(self.btn_import)

        self.btn_export_both = QPushButton("Export to CSV & JSON")
        self.btn_export_both.clicked.connect(lambda: self._export_layout(fmt="csv"))
        top.addWidget(self.btn_export_both)

        self.btn_png = QPushButton("PNG")
        self.btn_svg = QPushButton("SVG")
        self.btn_png.clicked.connect(self.export_png)
        self.btn_svg.clicked.connect(self.export_svg)
        top.addWidget(self.btn_png)
        top.addWidget(self.btn_svg)

        lay.addLayout(top)
        lay.addWidget(self.view, 1)

        self.status = QLabel("Mode: Cage tool. Click cells to add/remove cages.")
        self.status.setStyleSheet("color:#9aa7b0;")
        lay.addWidget(self.status)

        self.setStyleSheet("""
            QLabel { color: #cfd8dc; }
            QPushButton { padding: 6px 10px; background:#1b2028; color:#cfd8dc; border:1px solid #2a3440; border-radius:6px; }
            QPushButton:hover { background:#222a35; }
            QPushButton:disabled { color:#6f7b86; }
            QLineEdit { padding: 3px; background:#141922; color:#e6eef2; border:1px solid #2a3440; border-radius:6px; }
            QSpinBox { padding-right: 20px; background:#141922; color:#e6eef2; border:1px solid #2a3440; border-radius:6px; }
            QSpinBox::up-button {
                subcontrol-origin: border; subcontrol-position: top right; width: 16px;
                border-left: 1px solid #2a3440; border-top: 1px solid #2a3440;
            }
            QSpinBox::down-button {
                subcontrol-origin: border; subcontrol-position: bottom right; width: 16px;
                border-left: 1px solid #2a3440; border-bottom: 1px solid #2a3440;
            }
        """)

        self._set_mode("cage")

    def _set_mode(self, mode: str):
        self.mode = mode
        if mode == "cage":
            self._tunnel_start = None
            self.view.setDragMode(QGraphicsView.NoDrag)
            self.status.setText("Mode: Cage tool. Click cells to add/remove cages.")
        elif mode == "tunnel":
            self.view.setDragMode(QGraphicsView.NoDrag)
            self.status.setText("Mode: Tunnel tool. Click a CAGE edge/corner (start), then click end cage or empty cell (dead end).")
        else:
            self._tunnel_start = None
            self.view.setDragMode(QGraphicsView.ScrollHandDrag)
            self.status.setText("Mode: Select tool. Click a tunnel to select; set antenna count; drag to pan.")

    # -----------------------------
    def _build_grid(self):
        size = self.GRID_N * self.CELL
        self.scene.setSceneRect(0, 0, size + self.MARGIN * 2, size + self.MARGIN * 2)

        bg = QGraphicsRectItem(self.scene.sceneRect())
        bg.setBrush(QBrush(QColor("#0f1115")))
        bg.setPen(QPen(Qt.NoPen))
        bg.setZValue(-100)
        self.scene.addItem(bg)

        for r in range(self.GRID_N):
            for c in range(self.GRID_N):
                x = self.MARGIN + c * self.CELL
                y = self.MARGIN + r * self.CELL
                rect = QRectF(x, y, self.CELL, self.CELL)
                cell = LayoutDialog.CellItem(self, r, c, rect)
                self.scene.addItem(cell)
                self.cells[(r, c)] = cell

    def _reset_scene(self):
        self.poll_timer.stop()
        self.countdown_timer.stop()
        self.scene.clear()
        self.cells = {}
        self.tunnels = []
        self.selected_tunnel = None
        self._tunnel_start = None
        self._active_editor_dot = None
        self.active_dot = None
        self._tunnel_serial = 1
        self._build_grid()

    # -----------------------------
    def _anchors_for_cell(self, cell: "LayoutDialog.CellItem") -> List[QPointF]:
        r = cell.rect()
        x0, y0, w, h = r.x(), r.y(), r.width(), r.height()
        return [
            QPointF(x0 + w/2, y0),         # 0 top mid
            QPointF(x0 + w/2, y0 + h),     # 1 bottom mid
            QPointF(x0, y0 + h/2),         # 2 left mid
            QPointF(x0 + w, y0 + h/2),     # 3 right mid
            QPointF(x0, y0),               # 4 TL
            QPointF(x0 + w, y0),           # 5 TR
            QPointF(x0, y0 + h),           # 6 BL
            QPointF(x0 + w, y0 + h),       # 7 BR
        ]

    def _nearest_anchor_idx(self, cell: "LayoutDialog.CellItem", scene_pos: QPointF) -> int:
        anchors = self._anchors_for_cell(cell)
        best_i = 0
        best_d = 1e18
        for i, a in enumerate(anchors):
            d = (a.x() - scene_pos.x())**2 + (a.y() - scene_pos.y())**2
            if d < best_d:
                best_d = d
                best_i = i
        return best_i

    def _nearest_anchor(self, cell: "LayoutDialog.CellItem", scene_pos: QPointF) -> QPointF:
        return self._anchors_for_cell(cell)[self._nearest_anchor_idx(cell, scene_pos)]

    def _anchor_towards(self, cell_from: "LayoutDialog.CellItem", cell_to: "LayoutDialog.CellItem") -> QPointF:
        dr = cell_to.r - cell_from.r
        dc = cell_to.c - cell_from.c
        r = cell_from.rect()
        x0, y0, w, h = r.x(), r.y(), r.width(), r.height()

        if abs(dc) > abs(dr):
            return QPointF(x0 + w, y0 + h/2) if dc > 0 else QPointF(x0, y0 + h/2)
        if abs(dr) > abs(dc):
            return QPointF(x0 + w/2, y0 + h) if dr > 0 else QPointF(x0 + w/2, y0)

        if dc >= 0 and dr >= 0:
            return QPointF(x0 + w, y0 + h)
        if dc >= 0 and dr < 0:
            return QPointF(x0 + w, y0)
        if dc < 0 and dr >= 0:
            return QPointF(x0, y0 + h)
        return QPointF(x0, y0)

    # -----------------------------
    def _apply_default_layout(self):
        for cid in ["3C", "5C", "3E", "5E"]:
            r, c = self._rc_from_id(cid)
            self.cells[(r, c)].set_cage(True)

        def add_tunnel(a: str, b: str, name: str):
            ra, ca = self._rc_from_id(a)
            rb, cb = self._rc_from_id(b)
            sa = self.cells[(ra, ca)]
            sb = self.cells[(rb, cb)]
            start_anchor = self._anchor_towards(sa, sb)
            end_anchor = self._anchor_towards(sb, sa)
            t = self._create_tunnel(sa, start_anchor, sb, end_anchor, dead_end=False, name=name)
            self.tunnels.append(t)

        add_tunnel("3C", "3E", "T1")
        add_tunnel("5C", "5E", "T2")
        add_tunnel("3C", "5C", "T3")
        add_tunnel("3E", "5E", "T4")

        self.status.setText("Default layout loaded. Use Cage/Tunnel/Select tools to modify.")

    # =========================
    # Graphics items
    # =========================
    class CellItem(QGraphicsRectItem):
        def __init__(self, dlg: "LayoutDialog", r: int, c: int, rect: QRectF):
            super().__init__(rect)
            self.dlg = dlg
            self.r = r
            self.c = c
            self.is_cage = False

            self.proxy: Optional[QGraphicsProxyWidget] = None
            self.cage_no: Optional[QLineEdit] = None
            self.cage_type: Optional[QLineEdit] = None

            self.anchor_items: List[QGraphicsEllipseItem] = []

            self.setBrush(QBrush(QColor("#171b22")))
            self.setPen(QPen(QColor("#2a3440"), 1))
            self.setZValue(0)
            self.setAcceptedMouseButtons(Qt.LeftButton)

        def mousePressEvent(self, event):
            self.dlg.on_cell_clicked(self, event.scenePos())
            event.accept()

        def _clear_anchors(self):
            for it in self.anchor_items:
                self.dlg.scene.removeItem(it)
            self.anchor_items = []

        def _make_anchors(self):
            self._clear_anchors()
            anchors = self.dlg._anchors_for_cell(self)

            rad = 5
            pen = QPen(QColor("#cfd8dc"), 2)
            brush = QBrush(QColor("#0f1115"))
            for a in anchors:
                e = QGraphicsEllipseItem(-rad, -rad, 2*rad, 2*rad)
                e.setPen(pen)
                e.setBrush(brush)
                e.setZValue(15)  # above cage fill, below tunnels
                e.setPos(a)
                e.setAcceptedMouseButtons(Qt.NoButton)  # don't steal clicks
                self.dlg.scene.addItem(e)
                self.anchor_items.append(e)

        def set_cage(self, enable: bool):
            if enable == self.is_cage:
                return
            self.is_cage = enable

            if enable:
                self.setZValue(10)

                # NEW style: fill like inputs, thick green outline
                self.setBrush(QBrush(QColor("#141922")))
                pen = QPen(QColor("#2ecc71"), 5)
                pen.setCosmetic(True)
                self.setPen(pen)
                self._make_anchors()

                # cage widget
                w = QWidget()
                w.setStyleSheet("""
                    QWidget { background:#141922; border:1px solid #2a3440; border-radius:10px; }
                    QLabel { color:#cfd8dc; font-weight:800; font-size:11px; }
                    QLineEdit { background:#141922; color:#e6eef2; border:1px solid #2a3440; border-radius:6px; padding:2px; font-size:11px; }
                """)
                v = QVBoxLayout(w)
                v.setContentsMargins(3, 3, 3, 3)
                v.setSpacing(4)

                lab = QLabel("Cage")
                lab.setAlignment(Qt.AlignHCenter)

                self.cage_no = QLineEdit()
                self.cage_no.setPlaceholderText("Cage no.")
                self.cage_type = QLineEdit()
                self.cage_type.setPlaceholderText("Cage type")

                # bigger and centered widget
                cw = int(self.rect().width() * 0.92)
                ch = int(self.rect().height() * 0.75)
                w.setFixedSize(cw, ch)

                v.addWidget(lab)
                v.addWidget(self.cage_no)
                v.addWidget(self.cage_type)

                self.proxy = QGraphicsProxyWidget()
                self.proxy.setWidget(w)
                self.proxy.setZValue(20)
                self.dlg.scene.addItem(self.proxy)

                # center in cell
                r = self.rect()
                px = r.x() + (r.width() - cw) / 2
                py = r.y() + (r.height() - ch) / 2
                self.proxy.setPos(px, py)
            else:
                self.setZValue(0)
                self.setBrush(QBrush(QColor("#171b22")))
                self.setPen(QPen(QColor("#2a3440"), 1))
                self._clear_anchors()
                if self.proxy is not None:
                    self.dlg.scene.removeItem(self.proxy)
                    self.proxy = None
                    self.cage_no = None
                    self.cage_type = None

        def cage_values(self) -> Tuple[str, str]:
            if not self.is_cage or not self.cage_no or not self.cage_type:
                return "", ""
            return self.cage_no.text().strip(), self.cage_type.text().strip()

        def center(self) -> QPointF:
            r = self.rect()
            return QPointF(r.x() + r.width()/2, r.y() + r.height()/2)

        def cell_id(self) -> str:
            return LayoutDialog.cell_id(self.r, self.c)

    class DotEllipse(QGraphicsEllipseItem):
        def __init__(self, dot: "LayoutDialog.AntennaDot"):
            super().__init__(-6, -6, 12, 12)
            self.dot = dot
            self.setAcceptedMouseButtons(Qt.LeftButton)

        def mousePressEvent(self, event):
            self.dot.on_clicked()
            event.accept()

    class AntennaDot:
        def __init__(self, dlg: "LayoutDialog", tunnel: "LayoutDialog.TunnelItem", idx: int, pos: QPointF):
            self.dlg = dlg
            self.tunnel = tunnel
            self.idx = idx
            self.pos = pos

            self.ellipse = LayoutDialog.DotEllipse(self)
            self.ellipse.setBrush(QBrush(QColor("#f1c40f")))
            self.ellipse.setPen(QPen(QColor("#f1c40f"), 1))
            self.ellipse.setZValue(70)
            self.ellipse.setPos(pos)
            self.dlg.scene.addItem(self.ellipse)

            self.label_item = QGraphicsTextItem("?")
            self.label_item.setDefaultTextColor(QColor("#e6eef2"))
            self.label_item.setZValue(75)
            self.label_item.setFont(self.dlg._font_antenna)  # bigger font
            self.dlg.scene.addItem(self.label_item)

            self.editor_proxy: Optional[QGraphicsProxyWidget] = None
            self.editor_edit: Optional[QLineEdit] = None
            self.editor_btn_check: Optional[QPushButton] = None
            self.editor_btn_accept: Optional[QPushButton] = None
            self.editor_btn_cancel: Optional[QPushButton] = None

            self.reposition()

        def remove(self):
            self.dlg._close_antenna_editor_if_open(self)
            self.dlg.scene.removeItem(self.ellipse)
            self.dlg.scene.removeItem(self.label_item)

        def get_label(self) -> str:
            return self.label_item.toPlainText().strip()

        def set_label(self, txt: str):
            txt = (txt or "").strip()
            self.label_item.setPlainText(txt if txt else "?")
            self.reposition()

        def on_clicked(self):
            self.dlg.open_antenna_editor(self)

        def reposition(self):
            start = self.tunnel.start_pt
            end = self.tunnel.end_pt
            dx, dy = end.x() - start.x(), end.y() - start.y()
            L = max(1e-6, (dx*dx + dy*dy) ** 0.5)
            nx, ny = -(dy / L), (dx / L)

            sign = 1 if (self.idx % 2 == 0) else -1
            off = 26
            lx = self.pos.x() + nx * off * sign
            ly = self.pos.y() + ny * off * sign
            self.label_item.setPos(lx, ly - 14)

            if self.editor_proxy is not None:
                ex = self.pos.x() + nx * (off + 56) * sign
                ey = self.pos.y() + ny * (off + 56) * sign
                self.editor_proxy.setPos(ex, ey - 18)

    class TunnelItem(QGraphicsLineItem):
        def __init__(
            self,
            dlg: "LayoutDialog",
            start_cell: "LayoutDialog.CellItem",
            start_pt: QPointF,
            end_cell: "LayoutDialog.CellItem",
            end_pt: QPointF,
            dead_end: bool,
            name: str,
        ):
            super().__init__(start_pt.x(), start_pt.y(), end_pt.x(), end_pt.y())
            self.dlg = dlg
            self.start_cell = start_cell
            self.end_cell = end_cell
            self.start_pt = start_pt
            self.end_pt = end_pt
            self.dead_end = dead_end
            self.name = (name or "").strip()

            self.antenna_count = 0
            self.dots: List["LayoutDialog.AntennaDot"] = []

            self.label_item = QGraphicsTextItem("")
            self.label_item.setDefaultTextColor(QColor("#e6eef2"))
            self.label_item.setZValue(60)
            f = QFont()
            f.setPointSize(10)
            f.setBold(True)
            self.label_item.setFont(f)
            self.dlg.scene.addItem(self.label_item)

            self._pen_normal = QPen(QColor("#cfd8dc"), 7)
            self._pen_selected = QPen(QColor("#e6eef2"), 9)
            if dead_end:
                self._pen_normal.setStyle(Qt.DashLine)
                self._pen_selected.setStyle(Qt.DashLine)

            self.setPen(self._pen_normal)
            self.setZValue(40)
            self.setFlag(QGraphicsLineItem.ItemIsSelectable, True)
            self.setAcceptedMouseButtons(Qt.LeftButton)

            self.dead_mark: List[QGraphicsLineItem] = []
            if dead_end:
                self._make_dead_mark()

            self.update_label_geometry()
            self.set_antenna_count(self.dlg.DEFAULT_ANTENNAS_PER_TUNNEL)

        def mousePressEvent(self, event):
            self.dlg.select_tunnel(self)
            super().mousePressEvent(event)

        def set_selected_style(self, selected: bool):
            self.setPen(self._pen_selected if selected else self._pen_normal)

        def _make_dead_mark(self):
            x, y = self.end_pt.x(), self.end_pt.y()
            a = 10
            p = QPen(QColor("#ff6b6b"), 4)
            l1 = QGraphicsLineItem(x - a, y - a, x + a, y + a)
            l2 = QGraphicsLineItem(x - a, y + a, x + a, y - a)
            l1.setPen(p); l2.setPen(p)
            l1.setZValue(45); l2.setZValue(45)
            self.dlg.scene.addItem(l1); self.dlg.scene.addItem(l2)
            self.dead_mark = [l1, l2]

        def update_label_geometry(self):
            title = self.name if self.name else "Tunnel"
            self.label_item.setPlainText(title)

            sx, sy = self.start_pt.x(), self.start_pt.y()
            ex, ey = self.end_pt.x(), self.end_pt.y()
            mx, my = (sx + ex)/2, (sy + ey)/2
            dx, dy = ex - sx, ey - sy
            L = max(1e-6, (dx*dx + dy*dy)**0.5)
            nx, ny = -(dy/L), (dx/L)

            off = 44
            self.label_item.setPos(mx + nx*off, my + ny*off)

        def set_name(self, name: str):
            self.name = (name or "").strip()
            self.update_label_geometry()

        def clear_dots(self):
            for d in self.dots:
                d.remove()
            self.dots = []

        def set_antenna_count(self, n: int):
            n = max(0, min(20, int(n)))
            old_labels = [d.get_label() for d in self.dots]

            self.antenna_count = n
            self.clear_dots()
            if n <= 0:
                return

            sx, sy = self.start_pt.x(), self.start_pt.y()
            ex, ey = self.end_pt.x(), self.end_pt.y()

            for i in range(1, n + 1):
                t = i / (n + 1)
                px = sx + (ex - sx) * t
                py = sy + (ey - sy) * t
                d = LayoutDialog.AntennaDot(self.dlg, self, i - 1, QPointF(px, py))
                self.dots.append(d)

            for i, d in enumerate(self.dots):
                if i < len(old_labels) and old_labels[i] and old_labels[i] != "?":
                    d.set_label(old_labels[i])
                else:
                    d.set_label("?")

        def remove(self):
            self.clear_dots()
            for m in self.dead_mark:
                self.dlg.scene.removeItem(m)
            self.dead_mark = []
            self.dlg.scene.removeItem(self.label_item)
            self.dlg.scene.removeItem(self)

    # -----------------------------
    def _create_tunnel(
        self,
        start_cell: CellItem,
        start_pt: QPointF,
        end_cell: CellItem,
        end_pt: QPointF,
        dead_end: bool,
        name: Optional[str] = None,
    ) -> TunnelItem:
        if not name:
            name = f"T{self._tunnel_serial}"
            self._tunnel_serial += 1
        t = LayoutDialog.TunnelItem(self, start_cell, start_pt, end_cell, end_pt, dead_end, name=name)
        self.scene.addItem(t)
        return t

    # -----------------------------
    def on_cell_clicked(self, cell: CellItem, scene_pos: QPointF):
        self._close_antenna_editor_if_open(None)

        if self.mode == "cage":
            cell.set_cage(not cell.is_cage)
            return

        if self.mode == "tunnel":
            if self._tunnel_start is None:
                if not cell.is_cage:
                    self.status.setText("Tunnel tool: start must be on a CAGE cell.")
                    return
                start_anchor = self._nearest_anchor(cell, scene_pos)
                self._tunnel_start = (cell, start_anchor)
                self.status.setText(f"Tunnel tool: start={cell.cell_id()}. Now click end cage or empty cell (dead end).")
                return

            start_cell, start_anchor = self._tunnel_start
            self._tunnel_start = None

            if cell.is_cage:
                end_anchor = self._nearest_anchor(cell, scene_pos)
                dead = False
            else:
                rect = cell.rect()
                end_anchor = QPointF(rect.x() + rect.width()/2, rect.y() + rect.height()/2)
                dead = True

            t = self._create_tunnel(start_cell, start_anchor, cell, end_anchor, dead_end=dead, name=None)
            self.tunnels.append(t)
            for tt in self.tunnels:
                tt.update()

            self.select_tunnel(t)
            self.status.setText("Tunnel created (default: 2 antennas). Tunnel tool remains active.")
            return

    # -----------------------------
    def select_tunnel(self, t: Optional[TunnelItem]):
        if self.selected_tunnel is not None:
            self.selected_tunnel.set_selected_style(False)

        self.selected_tunnel = t
        if t is None:
            self.tname_edit.setText("")
            return

        t.set_selected_style(True)

        self.ant_spin.blockSignals(True)
        self.ant_spin.setValue(int(t.antenna_count))
        self.ant_spin.blockSignals(False)

        self.tname_edit.blockSignals(True)
        self.tname_edit.setText(t.name)
        self.tname_edit.blockSignals(False)

    def _apply_antenna_count(self, val: int):
        if self.selected_tunnel is None:
            return
        self.selected_tunnel.set_antenna_count(int(val))

    def _delete_selected(self):
        if self.selected_tunnel is None:
            return
        t = self.selected_tunnel
        self.selected_tunnel = None
        try:
            self.tunnels.remove(t)
        except Exception:
            pass
        t.remove()
        self.tname_edit.setText("")

    def _on_tunnel_name_changed(self, txt: str):
        if self.selected_tunnel is None:
            return
        self.selected_tunnel.set_name(txt)

    # -----------------------------
    # Antenna editor
    # -----------------------------
    def _close_antenna_editor_if_open(self, keep_dot: Optional[AntennaDot]):
        if self._active_editor_dot is None:
            return
        if keep_dot is not None and self._active_editor_dot is keep_dot:
            return
        dot = self._active_editor_dot
        self._active_editor_dot = None
        if dot.editor_proxy is not None:
            self.scene.removeItem(dot.editor_proxy)
            dot.editor_proxy = None
            dot.editor_edit = None
            dot.editor_btn_check = None
            dot.editor_btn_accept = None
            dot.editor_btn_cancel = None

    def open_antenna_editor(self, dot: AntennaDot):
        self._close_antenna_editor_if_open(dot)

        if dot.editor_proxy is not None:
            return

        self._active_editor_dot = dot

        w = QWidget()
        w.setStyleSheet("""
            QWidget { background:#0f1115; border:1px solid #2a3440; border-radius:8px; }
            QLineEdit { padding:4px; background:#141922; color:#e6eef2; border:1px solid #2a3440; border-radius:6px; }
            QPushButton { padding:5px 10px; background:#1b2028; color:#cfd8dc; border:1px solid #2a3440; border-radius:6px; }
            QPushButton:hover { background:#222a35; }
        """)
        h = QHBoxLayout(w)
        h.setContentsMargins(8, 8, 8, 8)
        h.setSpacing(8)

        edit = QLineEdit(dot.get_label() if dot.get_label() != "?" else "")
        edit.setPlaceholderText("Antenna ID")
        edit.setFixedWidth(140)

        btn_check = QPushButton("Check")
        btn_check.setFixedWidth(80)

        btn_accept = QPushButton("Accept")
        btn_accept.setFixedWidth(90)

        btn_cancel = QPushButton("Cancel")
        btn_cancel.setFixedWidth(90)

        h.addWidget(edit)
        h.addWidget(btn_check)
        h.addWidget(btn_accept)
        h.addWidget(btn_cancel)

        proxy = QGraphicsProxyWidget()
        proxy.setWidget(w)
        proxy.setZValue(200)
        self.scene.addItem(proxy)

        dot.editor_proxy = proxy
        dot.editor_edit = edit
        dot.editor_btn_check = btn_check
        dot.editor_btn_accept = btn_accept
        dot.editor_btn_cancel = btn_cancel

        dot.reposition()

        def _accept():
            val = (dot.editor_edit.text() if dot.editor_edit else "").strip()
            dot.set_label(val)
            self._close_antenna_editor_if_open(None)

        def _cancel():
            self._close_antenna_editor_if_open(None)

        def _check():
            self.start_check_for_dot(dot)

        btn_accept.clicked.connect(_accept)
        btn_cancel.clicked.connect(_cancel)
        btn_check.clicked.connect(_check)

    # -----------------------------
    # "Check" (RFID listen)
    # -----------------------------
    def start_check_for_dot(self, dot: AntennaDot):
        if self.active_dot is not None:
            QMessageBox.information(self, "Busy", "Already waiting on another Check.")
            return

        daq_port = ""
        try:
            daq_port = self.main.daq_combo.currentText().strip()
        except Exception:
            daq_port = ""

        if not daq_port:
            QMessageBox.critical(self, "Missing DAQ COM", "Select the DAQ COM port in the main window.")
            return

        self.open_antenna_editor(dot)

        self.active_dot = dot
        self.check_remaining = self.CHECK_TIMEOUT_S

        if dot.editor_btn_check:
            dot.editor_btn_check.setText(f"Waiting:{self.check_remaining}")
            dot.editor_btn_check.setEnabled(False)

        self.status.setText("Check: waiting for RFID read to identify antenna/channel...")

        if self.main.board and self.main.board.is_running():
            start_counter, _, _ = self.main.board.get_last_rfid_event()
            self.check_start_counter = start_counter
            self._direct_thread = None
            self._direct_result = {"done": False, "ch": None}
            self.poll_timer.start()
            self.countdown_timer.start()
            return

        self.check_start_counter = None
        self._direct_result = {"done": False, "ch": None}
        self._direct_deadline = time.time() + self.CHECK_TIMEOUT_S

        def _worker():
            evt = read_one_rfid_event_from_port(daq_port, timeout_s=self.CHECK_TIMEOUT_S)
            self._direct_result["ch"] = (evt[0] if evt else None)
            self._direct_result["done"] = True

        self._direct_thread = threading.Thread(target=_worker, daemon=True)
        self._direct_thread.start()
        self.poll_timer.start()
        self.countdown_timer.start()

    def _finish_check(self, ok: bool, text: str, ch_value: Optional[str] = None):
        if self.active_dot is None:
            return
        dot = self.active_dot
        self.active_dot = None
        self.check_remaining = 0
        self.check_start_counter = None
        self._direct_thread = None
        self._direct_result = {"done": False, "ch": None}
        self._direct_deadline = 0.0
        self.poll_timer.stop()
        self.countdown_timer.stop()

        if dot.editor_btn_check:
            dot.editor_btn_check.setText("Check")
            dot.editor_btn_check.setEnabled(True)

        if ok and ch_value and dot.editor_edit:
            dot.editor_edit.setText(ch_value)

        self.status.setText(text)
        if not ok:
            QMessageBox.information(self, "Check", text)

    def _tick_check(self):
        if self.active_dot is None:
            self.countdown_timer.stop()
            return
        self.check_remaining = max(0, self.check_remaining - 1)
        if self.active_dot.editor_btn_check:
            self.active_dot.editor_btn_check.setText(f"Waiting:{self.check_remaining}")
        if self.check_remaining <= 0:
            self._finish_check(False, "Check: timeout (90s).")

    def _poll_check(self):
        if self.active_dot is None:
            self.poll_timer.stop()
            return

        if self.main.board and self.main.board.is_running() and self.check_start_counter is not None:
            cur_counter, tag, ch = self.main.board.get_last_rfid_event()
            if cur_counter > self.check_start_counter and ch:
                self._finish_check(True, f"Check: identified channel {ch} (tag {tag}).", ch_value=ch)
            return

        if self._direct_thread is not None:
            if self._direct_result.get("done", False):
                ch = self._direct_result.get("ch", None)
                if ch:
                    self._finish_check(True, f"Check: identified channel {ch}.", ch_value=ch)
                else:
                    self._finish_check(False, "Check: no RFID read detected (timeout).")
            else:
                if time.time() > self._direct_deadline + 5:
                    self._finish_check(False, "Check: timeout (thread).")

    # -----------------------------
    # Layout -> config logic
    # -----------------------------
    @staticmethod
    def _parse_first_int(s: str) -> Optional[int]:
        m = re.search(r"(\d+)", (s or ""))
        if not m:
            return None
        try:
            return int(m.group(1))
        except Exception:
            return None

    def _compute_layout_payload(self) -> Dict[str, Any]:
        """
        Produces:
          layout = {
            "cages": [{cell_id,cage_no,cage_type}],
            "tunnels": [{tunnel_no, name, start_cell_id, end_cell_id, dead_end, start_anchor_idx, end_anchor_idx, antennas:[...]}],
            "antenna_combinations": { "a_b": "cX_cY" or "cage_X" },
            "tunnels_map": { "cX_cY": "tunnel_N" }
          }
        """
        # --- Cages
        cages: List[Dict[str, Any]] = []
        cell_to_cage_no: Dict[str, str] = {}
        for cell in self.cells.values():
            if not cell.is_cage:
                continue
            cid = cell.cell_id()
            cage_no, cage_type = cell.cage_values()
            cage_no = (cage_no or "").strip()
            if not cage_no:
                raise ValueError(f"Cage at {cid} has empty 'Cage no.'. Fill it in (or remove the cage).")
            cages.append({"cell_id": cid, "cage_no": str(cage_no), "cage_type": str(cage_type or "")})
            cell_to_cage_no[cid] = str(cage_no)

        # --- Tunnels + end antennas
        tunnels_list: List[Dict[str, Any]] = []
        tunnel_end_ant: Dict[int, Tuple[str, str, str, str]] = {}
        #   idx -> (start_cell_id, end_cell_id, a_start, a_end) where a_* are *end* antennas (first/last dot)
        # NOTE: only for non-dead_end with both ends on cages

        # stable numbering: use digits in tunnel name if present; else sequential order in self.tunnels list
        used_nums = set()
        next_auto = 1
        for t in self.tunnels:
            tn = self._parse_first_int(t.name)  # e.g. "T4" -> 4
            if tn is None or tn in used_nums:
                while next_auto in used_nums:
                    next_auto += 1
                tn = next_auto
            used_nums.add(tn)

            # store enough for import/rebuild
            start_cid = t.start_cell.cell_id()
            end_cid = t.end_cell.cell_id()
            dead = bool(t.dead_end)
            # anchor idx (only meaningful if cell is cage, for dead end we keep None)
            start_anchor_idx = None
            end_anchor_idx = None
            try:
                # best-effort: snap to nearest anchor of those stored points
                start_anchor_idx = self._nearest_anchor_idx(t.start_cell, t.start_pt)
            except Exception:
                start_anchor_idx = None
            if not dead:
                try:
                    end_anchor_idx = self._nearest_anchor_idx(t.end_cell, t.end_pt)
                except Exception:
                    end_anchor_idx = None

            ants = [d.get_label() for d in t.dots]
            tunnels_list.append({
                "tunnel_no": int(tn),
                "name": str(t.name or f"T{tn}"),
                "start_cell_id": start_cid,
                "end_cell_id": end_cid,
                "dead_end": dead,
                "start_anchor_idx": start_anchor_idx,
                "end_anchor_idx": end_anchor_idx,
                "antenna_count": int(t.antenna_count),
                "antennas": ants,
            })

            # for config generation: only non-dead tunnels between two cages with >=2 antennas and labeled ends
            if dead:
                continue
            if start_cid not in cell_to_cage_no or end_cid not in cell_to_cage_no:
                continue

            if len(t.dots) < 2:
                raise ValueError(f"Tunnel '{t.name}' must have at least 2 antennas for export.")
            a_start = (t.dots[0].get_label() or "").strip()
            a_end = (t.dots[-1].get_label() or "").strip()
            if (not a_start) or a_start == "?" or (not a_end) or a_end == "?":
                raise ValueError(f"Tunnel '{t.name}' has unlabeled end antenna(s). Label the first and last antenna dots.")
            tunnel_end_ant[id(t)] = (start_cid, end_cid, a_start, a_end)

        # --- antenna_combinations + tunnels_map
        antenna_combinations: Dict[str, str] = {}
        tunnels_map: Dict[str, str] = {}

        # 1) tunnels: aStart_aEnd -> cX_cY, reverse
        # 2) tunnels_map: cX_cY -> tunnel_N, reverse
        # We must match tunnel_no by record
        tunnel_key_to_no: Dict[Tuple[str, str], int] = {}
        for rec in tunnels_list:
            if rec.get("dead_end"):
                continue
            sc, ec = rec["start_cell_id"], rec["end_cell_id"]
            if sc in cell_to_cage_no and ec in cell_to_cage_no:
                tunnel_key_to_no[(sc, ec)] = int(rec["tunnel_no"])

        # helper: find ends again from rec (more robust than id(t))
        for rec in tunnels_list:
            if rec.get("dead_end"):
                continue
            sc, ec = rec["start_cell_id"], rec["end_cell_id"]
            if sc not in cell_to_cage_no or ec not in cell_to_cage_no:
                continue
            ants = rec.get("antennas") or []
            if len(ants) < 2:
                continue
            a_start = (ants[0] or "").strip()
            a_end = (ants[-1] or "").strip()
            if not a_start or a_start == "?" or not a_end or a_end == "?":
                continue

            cA = cell_to_cage_no[sc]
            cB = cell_to_cage_no[ec]
            dir_ab = f"c{cA}_c{cB}"
            dir_ba = f"c{cB}_c{cA}"

            antenna_combinations[f"{a_start}_{a_end}"] = dir_ab
            antenna_combinations[f"{a_end}_{a_start}"] = dir_ba

            tn = int(rec["tunnel_no"])
            tunnels_map[dir_ab] = f"tunnel_{tn}"
            tunnels_map[dir_ba] = f"tunnel_{tn}"

        # 3) cages: generalized logic based on incident tunnels
        # For each cage: build (prox, opp) pairs for each incident tunnel
        cage_inc: Dict[str, List[Tuple[str, str]]] = {cid: [] for cid in cell_to_cage_no.keys()}
        for rec in tunnels_list:
            if rec.get("dead_end"):
                continue
            sc, ec = rec["start_cell_id"], rec["end_cell_id"]
            if sc not in cell_to_cage_no or ec not in cell_to_cage_no:
                continue
            ants = rec.get("antennas") or []
            if len(ants) < 2:
                continue
            a_sc = (ants[0] or "").strip()   # prox to start cage
            a_ec = (ants[-1] or "").strip()  # prox to end cage
            if not a_sc or a_sc == "?" or not a_ec or a_ec == "?":
                continue

            cage_inc[sc].append((a_sc, a_ec))  # prox=sc, opp=ec
            cage_inc[ec].append((a_ec, a_sc))  # prox=ec, opp=sc

        for cid, pairs in cage_inc.items():
            cage_no = cell_to_cage_no[cid]
            cage_label = f"cage_{cage_no}"

            # self on each prox
            for prox, _opp in pairs:
                antenna_combinations[f"{prox}_{prox}"] = cage_label

            # prox-prox pairs (all directions)
            prox_list = [p for p, _ in pairs]
            for i in range(len(prox_list)):
                for j in range(i + 1, len(prox_list)):
                    p_i = prox_list[i]
                    p_j = prox_list[j]
                    antenna_combinations[f"{p_i}_{p_j}"] = cage_label
                    antenna_combinations[f"{p_j}_{p_i}"] = cage_label

            # prox_i with opp_j for i != j (all directions)
            for i, (p_i, _o_i) in enumerate(pairs):
                for j, (_p_j, o_j) in enumerate(pairs):
                    if i == j:
                        continue
                    antenna_combinations[f"{p_i}_{o_j}"] = cage_label
                    antenna_combinations[f"{o_j}_{p_i}"] = cage_label

        return normalize_layout_for_config({
            "cages": cages,
            "tunnels": tunnels_list,
            "antenna_combinations_interp": antenna_combinations,
            "antenna_combinations": antenna_combinations,
            "tunnels_map": tunnels_map,
        })

    def _export_layout(self, fmt: str):
        try:
            layout_payload = self._compute_layout_payload()
            self.main.update_shared_config(layout=layout_payload, animals=None)
            csv_path, json_path = self.main.get_shared_config_paths()
            if fmt.lower() == "json":
                QMessageBox.information(self, "Exported", f"Updated JSON:\n{json_path}")
            else:
                QMessageBox.information(self, "Exported", f"Updated CSV:\n{csv_path}\n(and JSON:\n{json_path})")
        except Exception as e:
            QMessageBox.critical(self, "Export failed", str(e))


    def _on_preset_selected(self, _idx: int):
        name = (self.preset_combo.currentText() or "").strip()
        if name.startswith("Custom"):
            self._reset_scene()
            self.status.setText("Preset: Custom (empty).")
            return
        if "Classic" in name:
            self._apply_layout_from_cfg(CLASSIC_LAYOUT_PRESET_CFG)
            self.status.setText("Preset: Classic DeepEcoHAB loaded.")
            return
        if "Field" in name:
            self._apply_layout_from_cfg(FIELD_LAYOUT_PRESET_CFG)
            self.status.setText("Preset: Field DeepEcoHAB loaded.")
            return

    def _apply_layout_from_cfg(self, cfg: Dict[str, Any]):
        self._reset_scene()

        layout = normalize_layout_for_config(cfg.get("layout") or {})
        cages = [rec for _key, rec in _layout_cage_records(layout)]
        tunnels = [rec for _key, rec in _layout_tunnel_records(layout)]

        # cages
        for c in cages:
            cid = (c.get("cell_id") or "").strip()
            if not cid:
                continue
            try:
                r, cc = self._rc_from_id(cid)
                cell = self.cells[(r, cc)]
            except Exception:
                continue
            cell.set_cage(True)
            if cell.cage_no:
                cell.cage_no.setText(str(c.get("cage_no") or ""))
            if cell.cage_type:
                cell.cage_type.setText(str(c.get("cage_type") or ""))

        # tunnels
        max_tn = 0
        for t in tunnels:
            try:
                scid = (t.get("start_cell_id") or "").strip()
                ecid = (t.get("end_cell_id") or "").strip()
                if not scid or not ecid:
                    continue
                rs, cs = self._rc_from_id(scid)
                re_, ce_ = self._rc_from_id(ecid)
                start_cell = self.cells[(rs, cs)]
                end_cell = self.cells[(re_, ce_)]
                dead = bool(t.get("dead_end"))

                sidx = t.get("start_anchor_idx", None)
                eidx = t.get("end_anchor_idx", None)
                if sidx is None:
                    spt = start_cell.center()
                else:
                    anchors = self._anchors_for_cell(start_cell)
                    spt = anchors[int(sidx)] if 0 <= int(sidx) < len(anchors) else start_cell.center()

                if dead:
                    ept = end_cell.center()
                else:
                    if eidx is None:
                        ept = end_cell.center()
                    else:
                        anchors = self._anchors_for_cell(end_cell)
                        ept = anchors[int(eidx)] if 0 <= int(eidx) < len(anchors) else end_cell.center()

                name = str(t.get("name") or "")
                tn = int(t.get("tunnel_no") or (self._parse_first_int(name) or 0) or 0)
                if tn > 0:
                    max_tn = max(max_tn, tn)
                if not name:
                    name = f"T{tn}" if tn > 0 else f"T{self._tunnel_serial}"

                new_t = self._create_tunnel(start_cell, spt, end_cell, ept, dead_end=dead, name=name)
                self.tunnels.append(new_t)

                ants = t.get("antennas") or []
                if isinstance(ants, list) and len(ants) > 0:
                    new_t.set_antenna_count(len(ants))
                    for i, lab in enumerate(ants):
                        if i < len(new_t.dots):
                            new_t.dots[i].set_label(str(lab or "").strip())
                else:
                    # fall back
                    n = int(t.get("antenna_count") or 0) or self.DEFAULT_ANTENNAS_PER_TUNNEL
                    new_t.set_antenna_count(n)

            except Exception:
                continue

        # keep serial in sync
        if max_tn > 0:
            self._tunnel_serial = max_tn + 1

        self.status.setText("Imported layout from JSON.")

    def _import_json(self):
        path, _ = QFileDialog.getOpenFileName(self, "Import config JSON", self.main.save_edit.text(), "JSON files (*.json)")
        if not path:
            return
        cfg = self.main.load_shared_config_from_file(Path(path))
        if not cfg:
            QMessageBox.critical(self, "Import failed", "Could not read JSON.")
            return

        # Do not overwrite the current experiment name or output directory when loading a template.
        self._apply_layout_from_cfg(cfg)
        QMessageBox.information(self, "Imported", f"Imported layout from:\n{Path(path).name}")

    # -----------------------------
    # PNG/SVG export
    # -----------------------------
    def export_png(self):
        try:
            save_dir = Path(self.main.save_edit.text().strip() or str(Path.cwd()))
            save_dir.mkdir(parents=True, exist_ok=True)
            exp = safe_slug(self.main.exp_edit.text(), "experiment")
            stamp = time.strftime("%Y%m%d_%H%M%S")
            out = save_dir / f"{exp}_{stamp}_layout.png"

            br = self.scene.itemsBoundingRect().adjusted(-30, -30, 30, 30)
            scale = 2
            img = QImage(int(br.width() * scale), int(br.height() * scale), QImage.Format_ARGB32)
            img.fill(QColor("#0f1115"))
            p = QPainter(img)
            p.setRenderHints(QPainter.Antialiasing | QPainter.TextAntialiasing)
            p.scale(scale, scale)
            self.scene.render(p, QRectF(0, 0, br.width(), br.height()), br)
            p.end()

            img.save(str(out))
            QMessageBox.information(self, "Export PNG", f"Saved:\n{out}")
        except Exception as e:
            QMessageBox.critical(self, "Export PNG failed", str(e))

    def export_svg(self):
        if not _HAS_SVG:
            QMessageBox.critical(self, "SVG unavailable", "PySide6 QtSvg is not available in this environment.")
            return
        try:
            save_dir = Path(self.main.save_edit.text().strip() or str(Path.cwd()))
            save_dir.mkdir(parents=True, exist_ok=True)
            exp = safe_slug(self.main.exp_edit.text(), "experiment")
            stamp = time.strftime("%Y%m%d_%H%M%S")
            out = save_dir / f"{exp}_{stamp}_layout.svg"

            br = self.scene.itemsBoundingRect().adjusted(-30, -30, 30, 30)
            gen = QSvgGenerator()
            gen.setFileName(str(out))
            gen.setSize(br.size().toSize())
            gen.setViewBox(br)

            p = QPainter(gen)
            p.setRenderHints(QPainter.Antialiasing | QPainter.TextAntialiasing)
            p.fillRect(br, QColor("#0f1115"))
            self.scene.render(p, br, br)
            p.end()

            QMessageBox.information(self, "Export SVG", f"Saved:\n{out}")
        except Exception as e:
            QMessageBox.critical(self, "Export SVG failed", str(e))

    def closeEvent(self, event):
        if self.active_dot is not None:
            self._finish_check(False, "Check cancelled (window closed).")
        super().closeEvent(event)


# -----------------------------
# Main Window (with shared config merge)
# -----------------------------
class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("DeepEcoHAB | DAQ + External-Trigger Camera (Qt)")
        self.resize(1500, 980)  # slightly taller so Tag totals can be taller without shrinking antenna box

        self.board: Optional[BoardAcquisition] = None
        self.camera: Optional[CameraWorker] = None
        self.trig = ArduinoTrigger()
        self.env = EnvironmentalController()
        self._env_log_path: Optional[Path] = None
        self._last_trigger_init_iso: str = ""
        self._experiment_started_iso: str = ""

        self.running = False
        self.stop_requested = False
        self.current_prefix: Optional[str] = None

        self._trigger_guard = False
        self.animal_dialog: Optional[AnimalDetailsDialog] = None
        self.layout_dialog: Optional[LayoutDialog] = None

        # Imported JSON files are used only as templates. Future exports always use the
        # currently selected output directory + current experiment name.

        self._build_ui()
        self._refresh_ports()

        self.t_preview = QTimer(self); self.t_preview.setInterval(100); self.t_preview.timeout.connect(self._update_preview)
        self.t_ind = QTimer(self); self.t_ind.setInterval(200); self.t_ind.timeout.connect(self._update_indicators)
        self.t_tick = QTimer(self); self.t_tick.setInterval(500); self.t_tick.timeout.connect(self._tick)
        self.t_tags = QTimer(self); self.t_tags.setInterval(60_000); self.t_tags.timeout.connect(self._refresh_tag_table)

        self.t_preview.start()
        self.t_ind.start()
        self.t_tick.start()
        self.t_tags.start()

    # -----------------------------
    # Shared config paths + merge writer
    # -----------------------------
    def get_shared_config_paths(self) -> Tuple[Path, Path]:
        """Return the active experiment config paths.

        Important: imported JSON files are templates only. We always save to the directory
        currently visible in the GUI, using the current experiment name.
        """
        save_dir = Path(self.save_edit.text().strip() or str(Path.cwd()))
        save_dir.mkdir(parents=True, exist_ok=True)
        exp = safe_slug(self.exp_edit.text(), "experiment")
        json_path = save_dir / f"{exp}_config.json"
        csv_path = save_dir / f"{exp}_config.csv"
        return csv_path, json_path

    def load_shared_config_from_file(self, json_path: Path) -> Optional[Dict[str, Any]]:
        cfg = read_config_json(json_path)
        if not cfg:
            return None

        # Imported JSON is treated as a template only. Do not change export targets and
        # do not overwrite the current experiment directory/name.
        if "experiment" not in cfg:
            cfg["experiment"] = {"name": self.exp_edit.text().strip(), "updated": dt.datetime.now().isoformat(), "start_datetime": "", "recording_timezone": "Europe/Warsaw"}
        if "layout" not in cfg:
            cfg["layout"] = {"cages": {}, "tunnels": {}, "antenna_combinations_interp": {}, "antenna_combinations": {}, "tunnels_map": {}}
        if "animals" not in cfg:
            cfg["animals"] = {"n_mice": 0, "rows": {}}

        if "environment" not in cfg:
            cfg["environment"] = {}
        if "hardware_trigger" not in cfg:
            cfg["hardware_trigger"] = {}
        if "camera_trigger" not in cfg:
            cfg["camera_trigger"] = {}

        return cfg

    def update_shared_config(self, layout: Optional[Dict[str, Any]], animals: Optional[Dict[str, Any]], environment: Optional[Dict[str, Any]] = None, hardware_trigger: Optional[Dict[str, Any]] = None) -> None:
        csv_path, json_path = self.get_shared_config_paths()

        cfg = read_config_json(json_path)
        if not cfg:
            cfg = new_config_skeleton(self.exp_edit.text().strip())

        # normalize skeleton while keeping unknown/extra fields from older exports
        if "experiment" not in cfg:
            cfg["experiment"] = {}
        if "layout" not in cfg:
            cfg["layout"] = {}
        if "animals" not in cfg:
            cfg["animals"] = {"n_mice": 0, "rows": {}}
        if "environment" not in cfg:
            cfg["environment"] = {}
        if "hardware_trigger" not in cfg:
            cfg["hardware_trigger"] = {}
        if "camera_trigger" not in cfg:
            cfg["camera_trigger"] = {}

        # update experiment meta
        cfg["experiment"]["name"] = self.exp_edit.text().strip()
        cfg["experiment"]["updated"] = dt.datetime.now().isoformat()
        cfg["experiment"].setdefault("start_datetime", getattr(self, "_experiment_started_iso", ""))
        if getattr(self, "_experiment_started_iso", ""):
            cfg["experiment"]["start_datetime"] = getattr(self, "_experiment_started_iso", "")
        cfg["experiment"].setdefault("recording_timezone", "Europe/Warsaw")

        if layout is not None:
            layout_norm = normalize_layout_for_config(layout)
            cfg["layout"] = cfg.get("layout", {}) or {}
            for k, v in layout_norm.items():
                cfg["layout"][k] = v
            cfg["layout"] = normalize_layout_for_config(cfg["layout"])

        if animals is not None:
            cfg["animals"] = normalize_animals_for_config(animals)
        else:
            cfg["animals"] = normalize_animals_for_config(cfg.get("animals") or {})

        if environment is not None:
            cfg["environment"] = environment

        if hardware_trigger is not None:
            cfg["hardware_trigger"] = hardware_trigger

        # Always keep the acquisition camera-trigger settings in the shared config.
        try:
            cfg["camera_trigger"] = self._camera_trigger_payload()
        except Exception:
            pass

        # Normalize layout even when only animals/env/trigger are exported, so old target files are upgraded.
        cfg["layout"] = normalize_layout_for_config(cfg.get("layout") or {})

        # write JSON + CSV (CSV mirrors the combined v11 JSON hierarchy)
        write_config_json(cfg, json_path)
        write_config_csv(cfg, csv_path)

    # -----------------------------
    # UI
    # -----------------------------
    def _build_ui(self):
        root = QWidget()
        self.setCentralWidget(root)
        main_v = QVBoxLayout(root)
        main_v.setContentsMargins(10, 10, 10, 10)
        main_v.setSpacing(10)

        # --- Top row: Acquisition + Tag totals
        top_row = QHBoxLayout()
        top_row.setSpacing(10)
        main_v.addLayout(top_row)

        self.acq_box = QGroupBox("Acquisition")
        top_row.addWidget(self.acq_box, 3)

        # Tag totals moved next to Acquisition (does not steal vertical space from antenna activity)
        self.tags_box = QGroupBox("Tag totals (refreshes every 60 seconds)")
        self.tags_box.setStyleSheet("QGroupBox { background:#151a23; } QLabel { background: transparent; } QWidget { background: transparent; }")
        self.tags_box.setFixedHeight(320)
        self.tags_box.setMinimumWidth(420)
        tv = QVBoxLayout(self.tags_box)
        tv.setContentsMargins(8, 8, 8, 8)
        tv.setSpacing(6)

        top_tags = QWidget()
        th2 = QHBoxLayout(top_tags); th2.setContentsMargins(0, 0, 0, 0); th2.setSpacing(10)
        self.tags_info = QLabel("Tag totals: (not yet refreshed)")
        self.btn_export_tags = QPushButton("Save unique tags to CSV")
        self.btn_export_tags.clicked.connect(self._save_unique_tags_csv)
        th2.addWidget(self.tags_info, 1)
        th2.addWidget(self.btn_export_tags)
        tv.addWidget(top_tags)

        self.tag_table = QTableWidget(0, 2)
        self.tag_table.setHorizontalHeaderLabels(["TAG", "Count"])
        self.tag_table.setSortingEnabled(False)
        self.tag_table.setEditTriggers(QTableWidget.NoEditTriggers)
        self.tag_table.setSelectionBehavior(QTableWidget.SelectRows)
        self.tag_table.setColumnWidth(0, 240)
        self.tag_table.setColumnWidth(1, 90)
        tv.addWidget(self.tag_table, 1)

        top_row.addWidget(self.tags_box, 2)

        acq_form = QFormLayout(self.acq_box)
        acq_form.setLabelAlignment(Qt.AlignRight)

        self.exp_edit = QLineEdit("DeepEcoHAB")
        self.exp_edit.setMaximumWidth(320)

        exp_row = QWidget()
        eh = QHBoxLayout(exp_row); eh.setContentsMargins(0, 0, 0, 0); eh.setSpacing(10)
        eh.addWidget(self.exp_edit)

        self.btn_animals = QPushButton("Animal details")
        self.btn_animals.clicked.connect(self._open_animal_details)
        eh.addWidget(self.btn_animals)

        self.btn_layout = QPushButton("DeepEcoHAB layout")
        self.btn_layout.clicked.connect(self._open_layout)
        eh.addWidget(self.btn_layout)

        eh.addStretch(1)
        acq_form.addRow("Experiment name:", exp_row)

        daq_row = QWidget()
        daq_h = QHBoxLayout(daq_row); daq_h.setContentsMargins(0, 0, 0, 0); daq_h.setSpacing(8)
        self.daq_combo = QComboBox(); self.daq_combo.setMinimumWidth(140)
        self.btn_refresh = QPushButton("Refresh ports"); self.btn_refresh.clicked.connect(self._refresh_ports)
        daq_h.addWidget(self.daq_combo)
        daq_h.addWidget(self.btn_refresh)
        daq_h.addStretch(1)
        acq_form.addRow("DAQ COM:", daq_row)

        save_row = QWidget()
        save_h = QHBoxLayout(save_row); save_h.setContentsMargins(0, 0, 0, 0); save_h.setSpacing(8)
        self.save_edit = QLineEdit(str(Path.cwd()))
        self.btn_browse = QPushButton("Browse"); self.btn_browse.clicked.connect(self._browse_dir)
        save_h.addWidget(self.save_edit, 1)
        save_h.addWidget(self.btn_browse)
        acq_form.addRow("Save dir:", save_row)

        res_row = QWidget()
        res_h = QHBoxLayout(res_row); res_h.setContentsMargins(0, 0, 0, 0); res_h.setSpacing(8)
        self.w_spin = make_spinbox(64, 8192, 1280)
        self.h_spin = make_spinbox(64, 8192, 1280)
        self.resize_chk = QCheckBox("Software resize if camera can't")
        res_h.addWidget(QLabel("W")); res_h.addWidget(self.w_spin)
        res_h.addWidget(QLabel("H")); res_h.addWidget(self.h_spin)
        res_h.addSpacing(10)
        res_h.addWidget(self.resize_chk)
        res_h.addStretch(1)
        acq_form.addRow("Resolution:", res_row)

        self.enc_combo = QComboBox()
        self.enc_combo.setEditable(True)
        self.enc_combo.lineEdit().setReadOnly(True)
        self.enc_combo.setInsertPolicy(QComboBox.NoInsert)
        model = QStandardItemModel()
        for opt in ENCODING_OPTIONS:
            it = QStandardItem(opt.full_label)
            it.setData(opt.short_label, Qt.UserRole)
            model.appendRow(it)
        self.enc_combo.setModel(model)
        self.enc_combo.setCurrentIndex(0)
        self._sync_encoding_display()
        self.enc_combo.currentIndexChanged.connect(self._sync_encoding_display)
        acq_form.addRow("Encoding:", self.enc_combo)

        misc_row = QWidget()
        misc_h = QHBoxLayout(misc_row); misc_h.setContentsMargins(0, 0, 0, 0); misc_h.setSpacing(10)
        self.preview_chk = QCheckBox("Preview"); self.preview_chk.setChecked(True)
        self.preview_fps = make_spinbox(1, 30, 10)
        self.seg_hours = make_spinbox(1, 24, 1)
        misc_h.addWidget(self.preview_chk)
        misc_h.addWidget(QLabel("Preview FPS")); misc_h.addWidget(self.preview_fps)
        misc_h.addSpacing(8)
        misc_h.addWidget(QLabel("Segment (hours)")); misc_h.addWidget(self.seg_hours)
        misc_h.addStretch(1)
        acq_form.addRow("Preview/Segment:", misc_row)

        cam_trig_row = QWidget()
        cam_trig_h = QHBoxLayout(cam_trig_row); cam_trig_h.setContentsMargins(0, 0, 0, 0); cam_trig_h.setSpacing(8)
        self.cam_force_trigger_chk = QCheckBox("Force external camera trigger")
        self.cam_force_trigger_chk.setChecked(True)
        self.cam_trigger_source_combo = QComboBox()
        self.cam_trigger_source_combo.addItems(["Line0", "Line1", "Line2", "Line3", "Counter0Start"])
        self.cam_trigger_source_combo.setCurrentText("Line0")
        self.cam_trigger_activation_combo = QComboBox()
        self.cam_trigger_activation_combo.addItems(["RisingEdge", "FallingEdge", "AnyEdge", "LevelHigh", "LevelLow"])
        self.cam_trigger_activation_combo.setCurrentText("RisingEdge")
        self.cam_trigger_delay_spin = make_spinbox(0, 1000000, 24)
        cam_trig_h.addWidget(self.cam_force_trigger_chk)
        cam_trig_h.addWidget(QLabel("Source")); cam_trig_h.addWidget(self.cam_trigger_source_combo)
        cam_trig_h.addWidget(QLabel("Edge")); cam_trig_h.addWidget(self.cam_trigger_activation_combo)
        cam_trig_h.addWidget(QLabel("Delay µs")); cam_trig_h.addWidget(self.cam_trigger_delay_spin)
        cam_trig_h.addStretch(1)
        acq_form.addRow("Camera trigger:", cam_trig_row)

        run_row = QWidget()
        run_h = QHBoxLayout(run_row); run_h.setContentsMargins(0, 0, 0, 0); run_h.setSpacing(10)
        self.btn_start = QPushButton("START (DAQ logs immediately; Camera waits for trigger)")
        self.btn_stop = QPushButton("STOP")
        self.btn_stop.setEnabled(False)
        self.btn_start.clicked.connect(self._on_start)
        self.btn_stop.clicked.connect(self._on_stop)
        run_h.addWidget(self.btn_start)
        run_h.addWidget(self.btn_stop)
        run_h.addStretch(1)
        acq_form.addRow("Run:", run_row)

        self.auto_export_envtrig_chk = QCheckBox("Automatically export trigger / environmental control to CSV & JSON upon START of the experiment")
        self.auto_export_envtrig_chk.setChecked(True)
        acq_form.addRow("", self.auto_export_envtrig_chk)

        # --- Middle: Trigger + Environmental
        mid_row = QHBoxLayout()
        mid_row.setSpacing(10)
        main_v.addLayout(mid_row)

        self.trig_box = QGroupBox("Hardware Trigger (Arduino)")
        self.env_box = QGroupBox("Environmental Control")
        mid_row.addWidget(self.trig_box, 1)
        mid_row.addWidget(self.env_box, 1)

        tform = QFormLayout(self.trig_box)
        tform.setLabelAlignment(Qt.AlignRight)

        trig_top = QWidget()
        th = QHBoxLayout(trig_top); th.setContentsMargins(0, 0, 0, 0); th.setSpacing(8)
        self.ard_combo = QComboBox(); self.ard_combo.setMinimumWidth(160)
        self.freq_spin = make_spinbox(1, 1000, 30)
        self.pulse_spin = make_spinbox(1, 1000, 4)
        th.addWidget(QLabel("COM")); th.addWidget(self.ard_combo)
        th.addSpacing(8)
        th.addWidget(QLabel("Hz")); th.addWidget(self.freq_spin)
        th.addSpacing(8)
        th.addWidget(QLabel("Pulse ms")); th.addWidget(self.pulse_spin)
        th.addStretch(1)
        tform.addRow("Connection:", trig_top)

        mode_row = QWidget()
        mh = QHBoxLayout(mode_row); mh.setContentsMargins(0, 0, 0, 0); mh.setSpacing(10)
        self.rb_count = QRadioButton("Count mode")
        self.rb_time = QRadioButton("Duration mode (DD:HH:MM)")
        self.rb_count.setChecked(True)
        self.mode_group = QButtonGroup(self)
        self.mode_group.addButton(self.rb_count, 0)
        self.mode_group.addButton(self.rb_time, 1)
        self.mode_group.buttonToggled.connect(self._trigger_mode_changed)
        mh.addWidget(self.rb_count); mh.addWidget(self.rb_time); mh.addStretch(1)
        tform.addRow("Mode:", mode_row)

        cd_row = QWidget()
        cd = QHBoxLayout(cd_row); cd.setContentsMargins(0, 0, 0, 0); cd.setSpacing(8)
        self.count_spin = make_spinbox(1, 2_000_000_000, 110000)
        self.dd_spin = make_spinbox(0, 365, 0)
        self.hh_spin = make_spinbox(0, 23, 0)
        self.mm_spin = make_spinbox(0, 59, 0)
        cd.addWidget(QLabel("Count")); cd.addWidget(self.count_spin)
        cd.addSpacing(10)
        cd.addWidget(QLabel("DD")); cd.addWidget(self.dd_spin)
        cd.addWidget(QLabel("HH")); cd.addWidget(self.hh_spin)
        cd.addWidget(QLabel("MM")); cd.addWidget(self.mm_spin)
        cd.addStretch(1)
        tform.addRow("Train length:", cd_row)

        self.btn_trigger = QPushButton("Initialize hardware trigger")
        self.btn_trigger.clicked.connect(self._on_trigger)
        tform.addRow("Action:", self.btn_trigger)

        self.trig_autostart_chk = QCheckBox("Start upon acquisition START (5s delay)")
        tform.addRow("Auto-start:", self.trig_autostart_chk)

        self.trig_status = QLabel("Trigger: Idle.")
        tform.addRow("Status:", self.trig_status)

        trig_exp = QWidget()
        te = QHBoxLayout(trig_exp); te.setContentsMargins(0, 0, 0, 0); te.setSpacing(8)
        self.btn_trig_export_both = QPushButton("Export to CSV & JSON")
        te.addWidget(self.btn_trig_export_both)
        te.addStretch(1)
        tform.addRow("Export:", trig_exp)

        self.btn_trig_export_both.clicked.connect(lambda: self._export_env_trigger(fmt="csv", which="trigger"))


        self.freq_spin.valueChanged.connect(self._trigger_params_changed)
        self.count_spin.valueChanged.connect(self._trigger_params_changed)
        self.dd_spin.valueChanged.connect(self._trigger_params_changed)
        self.hh_spin.valueChanged.connect(self._trigger_params_changed)
        self.mm_spin.valueChanged.connect(self._trigger_params_changed)

        self._apply_trigger_mode_state()
        self._trigger_params_changed()

        env_form = QFormLayout()
        env_form.setLabelAlignment(Qt.AlignRight)

        # Connection row
        env_conn = QWidget()
        eh = QHBoxLayout(env_conn); eh.setContentsMargins(0, 0, 0, 0); eh.setSpacing(8)
        self.env_combo = QComboBox(); self.env_combo.setMinimumWidth(160)
        eh.addWidget(QLabel("COM")); eh.addWidget(self.env_combo)
        eh.addStretch(1)
        env_form.addRow("Connection:", env_conn)

        # Light phase times
        phase_row = QWidget()
        ph = QHBoxLayout(phase_row); ph.setContentsMargins(0, 0, 0, 0); ph.setSpacing(8)
        self.env_light_start = QLineEdit("00:00"); self.env_light_start.setMaximumWidth(80)
        self.env_light_end = QLineEdit("12:00"); self.env_light_end.setMaximumWidth(80)
        ph.addWidget(QLabel("Start (HH:MM)")); ph.addWidget(self.env_light_start)
        ph.addSpacing(10)
        ph.addWidget(QLabel("End (HH:MM)")); ph.addWidget(self.env_light_end)
        ph.addStretch(1)
        env_form.addRow("Light phase:", phase_row)

        # Logging interval
        self.env_interval_spin = make_spinbox(1, 1440, 5)
        env_form.addRow("Log interval (min):", self.env_interval_spin)

        # Phase switch extra
        self.env_phase_switch_chk = QCheckBox("Log extra record at phase switch")
        env_form.addRow("", self.env_phase_switch_chk)

        self.env_autostart_chk = QCheckBox("Start upon acquisition START (5s delay)")
        env_form.addRow("Auto-start:", self.env_autostart_chk)

        # Actions
        env_btns = QWidget()
        bh = QHBoxLayout(env_btns); bh.setContentsMargins(0, 0, 0, 0); bh.setSpacing(8)
        self.btn_env_start = QPushButton("START env")
        self.btn_env_stop = QPushButton("STOP env"); self.btn_env_stop.setEnabled(False)
        self.btn_env_export_both = QPushButton("Export to CSV & JSON")
        bh.addWidget(self.btn_env_start)
        bh.addWidget(self.btn_env_stop)
        bh.addSpacing(10)
        bh.addWidget(self.btn_env_export_both)
        bh.addStretch(1)
        env_form.addRow("Action:", env_btns)

        # Status
        self.env_status = QLabel("Env: Idle.")
        self.env_last = QLabel("Last: -")
        self.env_last.setTextInteractionFlags(Qt.TextSelectableByMouse)
        env_form.addRow("Status:", self.env_status)
        env_form.addRow("Last reading:", self.env_last)

        env_v = QVBoxLayout(self.env_box)
        env_v.addLayout(env_form)
        env_v.addStretch(1)

        self.btn_env_start.clicked.connect(self._on_env_start)
        self.btn_env_stop.clicked.connect(self._on_env_stop)
        self.btn_env_export_both.clicked.connect(lambda: self._export_env_trigger(fmt="csv", which="env"))

        # --- Bottom: Preview | Antennas+Tags
        bottom = QSplitter(Qt.Horizontal)
        main_v.addWidget(bottom, 1)

        preview_box = QGroupBox("Live preview")
        self._preview_side = 300
        preview_box.setFixedWidth(self._preview_side + 40)
        preview_box.setSizePolicy(QSizePolicy.Fixed, QSizePolicy.Preferred)
        pv = QVBoxLayout(preview_box)
        pv.setContentsMargins(10, 10, 10, 10)
        pv.setSpacing(6)

        self.preview_label = QLabel("No frames yet.")
        self.preview_label.setAlignment(Qt.AlignCenter)
        self.preview_label.setStyleSheet("background:#111; color:#ddd; border:1px solid #444;")
        self.preview_label.setFixedSize(self._preview_side, self._preview_side)
        self.preview_label.setSizePolicy(QSizePolicy.Fixed, QSizePolicy.Fixed)
        pv.addWidget(self.preview_label, 0, Qt.AlignCenter)
        bottom.addWidget(preview_box)

        right = QWidget()
        rv = QVBoxLayout(right)
        rv.setContentsMargins(0, 0, 0, 0)
        rv.setSpacing(10)

        ind_scroll = QScrollArea()
        ind_scroll.setWidgetResizable(True)
        ind_host = QWidget()
        ind_scroll.setWidget(ind_host)
        ind_v = QVBoxLayout(ind_host)
        ind_v.setContentsMargins(6, 6, 6, 6)
        ind_v.setSpacing(8)

        ttl_group = QGroupBox("TTL channels")
        rfid_group = QGroupBox("RFID antennas")
        ind_v.addWidget(ttl_group)
        ind_v.addWidget(rfid_group)
        ind_v.addStretch(1)

        ttl_layout = QGridLayout(ttl_group)
        ttl_layout.setHorizontalSpacing(16)
        ttl_layout.setVerticalSpacing(8)
        self.indicators_ttl: Dict[str, Dict[str, QLabel]] = {}
        for idx, ch in enumerate(TTL_CHANNELS):
            cell = QWidget()
            h = QHBoxLayout(cell); h.setContentsMargins(0, 0, 0, 0); h.setSpacing(6)
            led = QLabel(); led.setStyleSheet(led_style(False))
            name = QLabel(f"TTL {ch}"); name.setMinimumWidth(52)
            tag = QLabel("TTL: -"); tag.setStyleSheet("font-family: Consolas, monospace;"); tag.setMinimumWidth(92)
            tag.setTextInteractionFlags(Qt.TextSelectableByMouse)
            h.addWidget(led); h.addWidget(name); h.addWidget(tag); h.addStretch(1)
            ttl_layout.addWidget(cell, 0, idx)
            self.indicators_ttl[ch] = {"led": led, "tag": tag}

        rfid_layout = QGridLayout(rfid_group)
        rfid_layout.setHorizontalSpacing(10)
        rfid_layout.setVerticalSpacing(8)
        self.indicators_rfid: Dict[str, Dict[str, QLabel]] = {}
        cols = 4
        for idx, ch in enumerate(RFID_CHANNELS):
            r = idx // cols
            c = idx % cols
            cell = QWidget()
            h = QHBoxLayout(cell); h.setContentsMargins(0, 0, 0, 0); h.setSpacing(6)
            led = QLabel(); led.setStyleSheet(led_style(False))
            name = QLabel(f"RFID {ch}"); name.setMinimumWidth(52)
            tag = QLabel("TAG: -"); tag.setStyleSheet("font-family: Consolas, monospace;"); tag.setMinimumWidth(120)
            tag.setTextInteractionFlags(Qt.TextSelectableByMouse)
            h.addWidget(led); h.addWidget(name); h.addWidget(tag); h.addStretch(1)
            rfid_layout.addWidget(cell, r, c)
            self.indicators_rfid[ch] = {"led": led, "tag": tag}

        ind_scroll_box = QGroupBox("Antenna activity")
        ind_scroll_box_v = QVBoxLayout(ind_scroll_box)
        ind_scroll_box_v.addWidget(ind_scroll)

        rv.addWidget(ind_scroll_box, 1)
        bottom.addWidget(right)
        bottom.setSizes([600, 900])  # fixed preview pane + activity panel

        self.statusBar().showMessage("Idle.")
        self.lbl_daq = QLabel("DAQ: -")
        self.lbl_cam = QLabel("Camera: -")
        self.statusBar().addPermanentWidget(self.lbl_daq, 1)
        self.statusBar().addPermanentWidget(self.lbl_cam, 1)

        self.setStyleSheet("""
            QGroupBox { font-weight: 600; }
            QPushButton { padding: 6px 10px; }
            QLineEdit, QComboBox, QDateEdit { padding: 3px; }
            QSpinBox { padding-right: 20px; }
            QSpinBox::up-button {
                subcontrol-origin: border; subcontrol-position: top right;
                width: 16px;
                border-left: 1px solid #e5e5e5;
                border-top: 1px solid #e5e5e5;
            }
            QSpinBox::down-button {
                subcontrol-origin: border; subcontrol-position: bottom right;
                width: 16px;
                border-left: 1px solid #e5e5e5;
                border-bottom: 1px solid #e5e5e5;
            }
        """)

    # -----------------------------
    # Dialog openers
    # -----------------------------
    def _open_animal_details(self):
        if self.animal_dialog is None or not self.animal_dialog.isVisible():
            self.animal_dialog = AnimalDetailsDialog(self)
            self.animal_dialog.show()
        else:
            self.animal_dialog.raise_()
            self.animal_dialog.activateWindow()

    def _open_layout(self):
        if self.layout_dialog is None or not self.layout_dialog.isVisible():
            self.layout_dialog = LayoutDialog(self)
            self.layout_dialog.show()
        else:
            self.layout_dialog.raise_()
            self.layout_dialog.activateWindow()

    # -----------------------------
    # Encoding label sync
    # -----------------------------
    def _sync_encoding_display(self):
        idx = self.enc_combo.currentIndex()
        if idx < 0:
            return
        short = self.enc_combo.model().item(idx).data(Qt.UserRole)
        self.enc_combo.lineEdit().setText(str(short))

    # -----------------------------
    # Trigger helpers
    # -----------------------------
    def _trigger_mode_changed(self, _btn, _checked: bool):
        self._apply_trigger_mode_state()
        self._trigger_params_changed()

    def _apply_trigger_mode_state(self):
        is_count_mode = self.rb_count.isChecked()
        self.count_spin.setEnabled(is_count_mode)
        self.dd_spin.setEnabled(not is_count_mode)
        self.hh_spin.setEnabled(not is_count_mode)
        self.mm_spin.setEnabled(not is_count_mode)

    def _trigger_params_changed(self):
        if self._trigger_guard:
            return
        self._trigger_guard = True
        try:
            freq = max(1, int(self.freq_spin.value()))
            if self.rb_count.isChecked():
                count = int(self.count_spin.value())
                seconds = count / freq
                dd, hh, mm = seconds_to_ddhhmm(seconds)
                self.dd_spin.setValue(dd)
                self.hh_spin.setValue(hh)
                self.mm_spin.setValue(mm)
            else:
                dd = int(self.dd_spin.value())
                hh = int(self.hh_spin.value())
                mm = int(self.mm_spin.value())
                seconds = ddhhmm_to_seconds(dd, hh, mm)
                count = int(round(seconds * freq))
                count = max(1, min(count, self.count_spin.maximum()))
                self.count_spin.setValue(count)
        finally:
            self._trigger_guard = False

    # -----------------------------
    # Ports
    # -----------------------------
    def _refresh_ports(self):
        ports = list_serial_ports()
        cur_daq = self.daq_combo.currentText()
        cur_ard = self.ard_combo.currentText()
        cur_env = self.env_combo.currentText() if hasattr(self, "env_combo") else ""

        self.daq_combo.clear()
        self.ard_combo.clear()
        if hasattr(self, "env_combo"):
            self.env_combo.clear()

        self.daq_combo.addItems(ports)
        self.ard_combo.addItems(ports)
        if hasattr(self, "env_combo"):
            self.env_combo.addItems(ports)

        if cur_daq in ports:
            self.daq_combo.setCurrentText(cur_daq)
        if cur_ard in ports:
            self.ard_combo.setCurrentText(cur_ard)
        if cur_env in ports and hasattr(self, "env_combo"):
            self.env_combo.setCurrentText(cur_env)

        self.statusBar().showMessage("Detected %d COM port(s)." % len(ports) if ports else "No COM ports detected.")

    def _browse_dir(self):
        d = QFileDialog.getExistingDirectory(self, "Select save directory", self.save_edit.text())
        if d:
            self.save_edit.setText(d)

    # -----------------------------
    # Run control
    # -----------------------------
    def _on_start(self):
        if self.running:
            return

        exp = safe_slug(self.exp_edit.text(), fallback="experiment")
        daq_port = self.daq_combo.currentText().strip()
        save_dir = self.save_edit.text().strip()

        if not daq_port:
            QMessageBox.critical(self, "Missing DAQ COM", "Select the DAQ COM port.")
            return
        if not save_dir:
            QMessageBox.critical(self, "Missing directory", "Choose a save directory.")
            return

        Path(save_dir).mkdir(parents=True, exist_ok=True)
        stamp = time.strftime("%Y%m%d_%H%M%S")
        prefix = f"{exp}_{stamp}"
        self.current_prefix = prefix
        self._experiment_started_iso = dt.datetime.now().isoformat()

        try:
            self.board = BoardAcquisition(port=daq_port, save_dir=save_dir, file_prefix=prefix, send_start_command=b"READC")
            self.board.start()

            seg_s = int(self.seg_hours.value()) * 3600
            w = int(self.w_spin.value())
            h = int(self.h_spin.value())

            self.camera = CameraWorker(
                save_dir=save_dir,
                file_prefix=prefix,
                fps=30,
                segment_time_s=seg_s,
                preview_enabled=bool(self.preview_chk.isChecked()),
                preview_max_fps=int(self.preview_fps.value()),
                target_width=w,
                target_height=h,
                software_resize_if_needed=bool(self.resize_chk.isChecked()),
                encoding_index=int(self.enc_combo.currentIndex()),
                force_external_trigger=bool(self.cam_force_trigger_chk.isChecked()),
                trigger_source=str(self.cam_trigger_source_combo.currentText()),
                trigger_activation=str(self.cam_trigger_activation_combo.currentText()),
                trigger_delay_us=float(self.cam_trigger_delay_spin.value()),
            )
            self.camera.start()

            # Create env log file for this experiment (filled when Environmental Control is running)
            try:
                self._env_log_path = Path(save_dir) / f"{prefix}_environment_log.csv"
                self.env.set_experiment_log_path(self._env_log_path)
            except Exception:
                pass

            self.running = True
            self.stop_requested = False
            self.btn_start.setEnabled(False)
            self.btn_stop.setEnabled(True)
            self.statusBar().showMessage(f"Running. Prefix={prefix} | DAQ logging now | Camera waiting for external triggers...")

            # Optional auto-start (5s delay after acquisition start)
            try:
                if hasattr(self, "env_autostart_chk") and self.env_autostart_chk.isChecked():
                    QTimer.singleShot(5000, self._autostart_env_after_delay)
                if hasattr(self, "trig_autostart_chk") and self.trig_autostart_chk.isChecked():
                    QTimer.singleShot(5000, self._autostart_trigger_after_delay)

                # Optional auto-export of trigger + environment details to the shared config
                if hasattr(self, "auto_export_envtrig_chk") and self.auto_export_envtrig_chk.isChecked():
                    self._auto_export_env_and_trigger(silent=True)
            except Exception:
                pass

        except Exception as e:
            self.current_prefix = None
            self.running = False
            self.btn_start.setEnabled(True)
            self.btn_stop.setEnabled(False)
            QMessageBox.critical(self, "Start failed", str(e))

    def _on_stop(self):
        if not self.running or self.stop_requested:
            return
        self.stop_requested = True
        self.statusBar().showMessage("Stop requested... closing camera/ffmpeg safely (EOF), stopping DAQ.")
        self.btn_stop.setEnabled(False)
        try:
            if self.camera:
                self.camera.request_stop()
        except Exception:
            pass
        try:
            if self.board:
                self.board.request_stop()
        except Exception:
            pass

    def _autostart_env_after_delay(self):
        if not self.running or self.stop_requested:
            return
        self._start_env(interactive=False)

    def _autostart_trigger_after_delay(self):
        if not self.running or self.stop_requested:
            return
        self._start_trigger(interactive=False)

    def _start_trigger(self, interactive: bool = True) -> bool:
        port = self.ard_combo.currentText().strip()
        if not port:
            if interactive:
                QMessageBox.critical(self, "Arduino COM missing", "Select the Arduino COM port.")
            else:
                self.statusBar().showMessage("Trigger auto-start skipped: no Arduino COM selected.")
            return False
        if self.trig.is_running():
            if interactive:
                QMessageBox.information(self, "Trigger busy", "Trigger is already running.")
            else:
                self.statusBar().showMessage("Trigger auto-start skipped: trigger already running.")
            return False

        freq = int(self.freq_spin.value())
        pulse = int(self.pulse_spin.value())

        # Determine count from selected mode
        if self.rb_count.isChecked():
            count = int(self.count_spin.value())
        else:
            dd = int(self.dd_spin.value())
            hh = int(self.hh_spin.value())
            mm = int(self.mm_spin.value())
            seconds = int(ddhhmm_to_seconds(dd, hh, mm))
            if seconds <= 0:
                if interactive:
                    QMessageBox.critical(self, "Bad trigger parameters", "Duration mode requires a duration > 0 (DD/HH/MM).")
                else:
                    self.statusBar().showMessage("Trigger auto-start skipped: duration=0.")
                return False
            count = int(round(seconds * max(freq, 1)))
            count = max(1, min(count, self.count_spin.maximum()))
            try:
                self.count_spin.setValue(count)
            except Exception:
                pass
        # Guard: pulse width must be shorter than the period (otherwise output can look like a constant HIGH)
        try:
            period_ms = 1000.0 / float(max(1, freq))
            if float(pulse) >= period_ms:
                msg = f"Pulse (ms) must be < period ({period_ms:.2f} ms at {freq} Hz).\nCurrent: {pulse} ms."
                if interactive:
                    QMessageBox.critical(self, "Bad trigger parameters", msg)
                else:
                    self.statusBar().showMessage("Trigger auto-start skipped: pulse>=period.")
                return False
        except Exception:
            pass


        if freq <= 0 or pulse <= 0 or count <= 0:
            if interactive:
                QMessageBox.critical(self, "Bad trigger parameters", "Freq, Pulse(ms), and Train length must be > 0.")
            else:
                self.statusBar().showMessage("Trigger auto-start skipped: invalid trigger parameters.")
            return False

        self._last_trigger_init_iso = dt.datetime.now().isoformat()
        self.trig.start_one_shot(port=port, baud=115200, freq_hz=freq, pulse_ms=pulse, count=count)
        if not interactive:
            self.statusBar().showMessage(f"Trigger auto-started on {port}.")
        return True

    def _on_trigger(self):
        if self._start_trigger(interactive=True):
            port = self.ard_combo.currentText().strip()
            self.statusBar().showMessage(f"Trigger started on {port}.")


    # -----------------------------
    # Environmental control (Arduino control box)
    # -----------------------------
    def _start_env(self, interactive: bool = True) -> bool:
        port = self.env_combo.currentText().strip()
        if not port:
            if interactive:
                QMessageBox.critical(self, "Env COM missing", "Select the Environmental Control Arduino COM port.")
            else:
                self.statusBar().showMessage("Env auto-start skipped: no Env COM selected.")
            return False
        if self.env.is_running():
            if interactive:
                QMessageBox.information(self, "Env busy", "Environmental controller is already running.")
            else:
                self.statusBar().showMessage("Env auto-start skipped: controller already running.")
            return False

        s = self.env_light_start.text().strip()
        e = self.env_light_end.text().strip()
        try:
            dt.datetime.strptime(s, "%H:%M")
            dt.datetime.strptime(e, "%H:%M")
        except Exception:
            if interactive:
                QMessageBox.critical(self, "Bad time format", "Light phase times must be HH:MM (e.g., 00:00).")
            else:
                self.statusBar().showMessage("Env auto-start skipped: bad HH:MM time format.")
            return False

        interval = int(self.env_interval_spin.value())
        phase_extra = bool(self.env_phase_switch_chk.isChecked())

        self._env_started_iso = dt.datetime.now().isoformat()
        self.env.start(
            port=port,
            baud=115200,
            light_start_hhmm=s,
            light_end_hhmm=e,
            log_interval_min=interval,
            phase_switch_extra=phase_extra,
        )

        self.btn_env_start.setEnabled(False)
        self.btn_env_stop.setEnabled(True)

        if not interactive:
            self.statusBar().showMessage(f"Environmental controller auto-started on {port}.")
        return True

    def _on_env_start(self):
        if self._start_env(interactive=True):
            port = self.env_combo.currentText().strip()
            self.statusBar().showMessage(f"Environmental controller started on {port}.")

    def _on_env_stop(self):

        if self.env.is_running():
            self.env.stop()
        self.btn_env_start.setEnabled(True)
        self.btn_env_stop.setEnabled(False)
        self.statusBar().showMessage("Environmental controller stop requested.")

    def _env_payload(self) -> Dict[str, Any]:
        rec = None
        try:
            with self.env._lock:
                rec = dict(self.env.last_record) if self.env.last_record else None
        except Exception:
            rec = None

        return {
            "com_port": self.env_combo.currentText().strip(),
            "auto_start_on_acquisition": bool(self.env_autostart_chk.isChecked()) if hasattr(self, "env_autostart_chk") else False,
            "light_start_hhmm": self.env_light_start.text().strip(),
            "light_end_hhmm": self.env_light_end.text().strip(),
            "log_interval_min": int(self.env_interval_spin.value()),
            "phase_switch_extra": bool(self.env_phase_switch_chk.isChecked()),
            "started_iso": getattr(self, "_env_started_iso", ""),
            "experiment_env_log_csv": str(self._env_log_path) if self._env_log_path else "",
            "last_record": rec or {},
        }

    def _camera_trigger_payload(self) -> Dict[str, Any]:
        return {
            "force_external_trigger": bool(self.cam_force_trigger_chk.isChecked()) if hasattr(self, "cam_force_trigger_chk") else True,
            "trigger_source": str(self.cam_trigger_source_combo.currentText()) if hasattr(self, "cam_trigger_source_combo") else "Line0",
            "trigger_activation": str(self.cam_trigger_activation_combo.currentText()) if hasattr(self, "cam_trigger_activation_combo") else "RisingEdge",
            "trigger_delay_us": int(self.cam_trigger_delay_spin.value()) if hasattr(self, "cam_trigger_delay_spin") else 24,
        }

    def _trigger_payload(self) -> Dict[str, Any]:
        freq = int(self.freq_spin.value())
        pulse = int(self.pulse_spin.value())
        count = int(self.count_spin.value())
        dd = int(self.dd_spin.value())
        hh = int(self.hh_spin.value())
        mm = int(self.mm_spin.value())
        mode = "count" if self.rb_count.isChecked() else "duration"

        if mode == "duration":
            train_s = int(ddhhmm_to_seconds(dd, hh, mm))
        else:
            train_s = float(count) / float(max(freq, 1))

        return {
            "com_port": self.ard_combo.currentText().strip(),
            "auto_start_on_acquisition": bool(self.trig_autostart_chk.isChecked()) if hasattr(self, "trig_autostart_chk") else False,
            "frequency_hz": freq,
            "pulse_ms": pulse,
            "mode": mode,
            "count": count,
            "duration_dd_hh_mm": {"dd": dd, "hh": hh, "mm": mm},
            "train_length_s": train_s,
            "train_length_dd_hh_mm": {"dd": seconds_to_ddhhmm(train_s)[0], "hh": seconds_to_ddhhmm(train_s)[1], "mm": seconds_to_ddhhmm(train_s)[2]},
            "initialized_iso": self._last_trigger_init_iso,
        }

    def _export_env_trigger(self, fmt: str, which: str):
        try:
            if which == "env":
                self.update_shared_config(layout=None, animals=None, environment=self._env_payload(), hardware_trigger=None)
            else:
                self.update_shared_config(layout=None, animals=None, environment=None, hardware_trigger=self._trigger_payload())
            csv_path, json_path = self.get_shared_config_paths()
            if fmt.lower() == "json":
                QMessageBox.information(self, "Exported", f"Updated JSON:\n{json_path}")
            else:
                QMessageBox.information(self, "Exported", f"Updated CSV:\n{csv_path}\n(and JSON:\n{json_path})")
        except Exception as e:
            QMessageBox.critical(self, "Export failed", str(e))

    def _auto_export_env_and_trigger(self, silent: bool = True):
        """Write environment + trigger blocks into the shared config (JSON+CSV).
        Used for the default auto-export on acquisition START.
        """
        try:
            self.update_shared_config(
                layout=None,
                animals=None,
                environment=self._env_payload(),
                hardware_trigger=self._trigger_payload(),
            )
            if not silent:
                csv_path, json_path = self.get_shared_config_paths()
                QMessageBox.information(self, "Exported", f"Updated CSV:\n{csv_path}\n(and JSON:\n{json_path})")
        except Exception as e:
            if not silent:
                QMessageBox.critical(self, "Export failed", str(e))


    # -----------------------------
    # Tags export
    # -----------------------------
    def _save_unique_tags_csv(self):
        try:
            if not self.board:
                QMessageBox.critical(self, "No DAQ", "DAQ is not running (no tags to export yet).")
                return

            tags = sorted(self.board.get_tag_counts_snapshot().keys())
            if not tags:
                QMessageBox.information(self, "No tags", "No RFID tags have been recorded yet.")
                return

            save_dir = Path(self.save_edit.text().strip() or str(Path.cwd()))
            save_dir.mkdir(parents=True, exist_ok=True)

            prefix = self.current_prefix or safe_slug(self.exp_edit.text(), "experiment")
            out_path = save_dir / f"{prefix}_unique_tags.csv"

            with open(out_path, "w", newline="", encoding="utf-8") as f:
                for t in tags:
                    f.write(f"{t}\n")

            QMessageBox.information(self, "Saved", f"Saved {len(tags)} unique tags to:\n{out_path}")

        except Exception as e:
            QMessageBox.critical(self, "Export failed", str(e))

    # -----------------------------
    # UI updates
    # -----------------------------
    def _tick(self):
        self.trig_status.setText(f"Trigger: {self.trig.status}")

        # Environmental status
        try:
            self.env_status.setText(f"Env: {self.env.status}")
            rec = None
            try:
                with self.env._lock:
                    rec = dict(self.env.last_record) if self.env.last_record else None
            except Exception:
                rec = None
            if rec:
                t = rec.get("temperature_C", float("nan"))
                h = rec.get("humidity_percent", float("nan"))
                ph = rec.get("phase", "")
                ts = rec.get("iso_datetime", "") or (rec.get("date", "") + " " + rec.get("time", ""))
                self.env_last.setText(f"{ts} | {ph} | {t:.2f} °C | {h:.2f} %RH")
            else:
                self.env_last.setText("Last: -")
        except Exception:
            pass


        if self.board:
            ttl_counts = ""
            try:
                ttl_counts = " | TTL " + " ".join([f"{ch}:{self.board.ttl_counts.get(ch, 0)}" for ch in TTL_CHANNELS])
            except Exception:
                ttl_counts = ""
            self.lbl_daq.setText(
                f"DAQ: ok={self.board.lines_ok} bad={self.board.lines_bad} dropped={self.board.lines_dropped} bytes={self.board.bytes_read}{ttl_counts} | {Path(self.board.data_file).name}"
            )
        else:
            self.lbl_daq.setText("DAQ: -")

        if self.camera:
            a = f"{self.camera.actual_width}x{self.camera.actual_height}" if self.camera.actual_width else "-"
            r = f"{self.camera.record_width}x{self.camera.record_height}" if self.camera.record_width else "-"
            wait = "WAITING_FOR_TRIGGER" if self.camera.waiting_for_first_frame else "RECORDING"
            trig = ""
            try:
                rb = (self.camera.camera_config_report or {}).get("readback", {})
                tm = rb.get("TriggerMode", "")
                ts = rb.get("TriggerSource", "")
                ta = rb.get("TriggerActivation", "")
                if tm or ts or ta:
                    trig = f" | trigger={tm}/{ts}/{ta}"
            except Exception:
                trig = ""
            self.lbl_cam.setText(
                f"Camera: {wait} | written={self.camera.frames_written} total={self.camera.frames_total} "
                f"dropped={self.camera.frames_dropped} q={self.camera.writer_queue_size} restarts={self.camera.ffmpeg_restarts} "
                f"| actual={a} record={r}{trig}"
            )
        else:
            self.lbl_cam.setText("Camera: -")

        if self.running:
            board_alive = self.board.is_running() if self.board else False
            cam_alive = self.camera.is_running() if self.camera else False
            if self.stop_requested and (not board_alive) and (not cam_alive):
                self.running = False
                self.stop_requested = False
                self.btn_start.setEnabled(True)
                self.btn_stop.setEnabled(False)
                warn = self.camera.last_error if (self.camera and self.camera.last_error) else ""
                self.statusBar().showMessage(f"Stopped with warning: {warn}" if warn else "Stopped.")

    def _update_indicators(self):
        if not self.board:
            for ch in TTL_CHANNELS:
                self.indicators_ttl[ch]["led"].setStyleSheet(led_style(False))
                self.indicators_ttl[ch]["tag"].setText("TTL: -")
            for ch in RFID_CHANNELS:
                self.indicators_rfid[ch]["led"].setStyleSheet(led_style(False))
                self.indicators_rfid[ch]["tag"].setText("TAG: -")
            return

        snap = self.board.get_status_snapshot()
        now = time.monotonic()

        for ch in TTL_CHANNELS:
            t_last = snap.get(ch, {}).get("t", 0.0)
            active = (now - t_last) <= TTL_ACTIVE_WINDOW_S
            self.indicators_ttl[ch]["led"].setStyleSheet(led_style(active))
            self.indicators_ttl[ch]["tag"].setText("TTL: pulse" if active else "TTL: -")

        for ch in RFID_CHANNELS:
            t_last = snap.get(ch, {}).get("t", 0.0)
            tag = snap.get(ch, {}).get("tag", "")
            active = (now - t_last) <= RFID_ACTIVE_WINDOW_S
            self.indicators_rfid[ch]["led"].setStyleSheet(led_style(active))
            self.indicators_rfid[ch]["tag"].setText(f"TAG: {tag if tag else '-'}")

    def _update_preview(self):
        if not (self.running and self.camera and self.preview_chk.isChecked()):
            return
        frame = self.camera.get_preview_frame()
        if frame is None:
            return
        fh, fw = frame.shape
        bpl = frame.strides[0]
        qimg = QImage(frame.data, fw, fh, bpl, QImage.Format_Grayscale8).copy()
        pix = QPixmap.fromImage(qimg)
        lw = self.preview_label.width()
        lh = self.preview_label.height()
        pix = pix.scaled(lw, lh, Qt.KeepAspectRatio, Qt.SmoothTransformation)
        self.preview_label.setPixmap(pix)
        self.preview_label.setText("")

    def _refresh_tag_table(self):
        if not self.board:
            self.tags_info.setText("Tag totals: (DAQ not running)")
            self.tag_table.setRowCount(0)
            return

        counts = self.board.get_tag_counts_snapshot()
        items = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)

        self.tag_table.setRowCount(len(items))
        for i, (tag, cnt) in enumerate(items):
            it_tag = QTableWidgetItem(str(tag))
            it_cnt = QTableWidgetItem(str(cnt))
            it_cnt.setTextAlignment(Qt.AlignRight | Qt.AlignVCenter)
            self.tag_table.setItem(i, 0, it_tag)
            self.tag_table.setItem(i, 1, it_cnt)

        self.tags_info.setText(
            f"Tag totals: refreshed {dt.datetime.now().strftime('%H:%M:%S')} | unique tags={len(items)}"
        )

    def closeEvent(self, event):
        try:
            if self.animal_dialog and self.animal_dialog.isVisible():
                self.animal_dialog.close()
        except Exception:
            pass
        try:
            if self.layout_dialog and self.layout_dialog.isVisible():
                self.layout_dialog.close()
        except Exception:
            pass
        try:
            if self.camera:
                self.camera.request_stop()
        except Exception:
            pass
        try:
            if self.board:
                self.board.request_stop()
        except Exception:
            pass
        try:
            if self.trig.is_running():
                self.trig.request_stop()
        except Exception:
            pass
        try:
            if hasattr(self, "env") and self.env.is_running():
                self.env.stop()
        except Exception:
            pass
        super().closeEvent(event)


# NOTE: original new-board launcher block removed; old-board GUI launches at end of file.

# -----------------------------
# Old board GUI main window
# -----------------------------
class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("DeepEcoHAB Old RFID DAQ")
        self.resize(1250, 820)
        self.board: Optional[OldBoardAcquisition] = None
        self.current_prefix: str = ""
        self._experiment_started_iso: str = ""
        self._last_tag_table_refresh = 0.0
        self._build_ui()
        self._refresh_ports()
        self._timer = QTimer(self)
        self._timer.timeout.connect(self._tick)
        self._timer.start(250)

    def _build_ui(self):
        central = QWidget()
        self.setCentralWidget(central)
        main_v = QVBoxLayout(central)
        main_v.setContentsMargins(10, 10, 10, 10)
        main_v.setSpacing(10)

        title = QLabel("DeepEcoHAB Old RFID DAQ")
        title.setStyleSheet("font-size:22px; font-weight:700; color:#e8eef6; background: transparent;")
        subtitle = QLabel("Legacy EcoHAB / old-board RFID recording — 8 RFID antennas, no TTL/camera/hardware trigger/environmental control")
        subtitle.setStyleSheet("color:#9aa4b2; background: transparent;")
        main_v.addWidget(title)
        main_v.addWidget(subtitle)

        top = QHBoxLayout()
        top.setSpacing(10)
        main_v.addLayout(top)

        self.acq_box = QGroupBox("Acquisition")
        top.addWidget(self.acq_box, 3)
        form = QFormLayout(self.acq_box)
        form.setLabelAlignment(Qt.AlignRight)

        self.exp_edit = QLineEdit("old_board_experiment")
        form.addRow("Experiment name:", self.exp_edit)

        row = QWidget(); h=QHBoxLayout(row); h.setContentsMargins(0,0,0,0); h.setSpacing(8)
        self.save_edit = QLineEdit(str(Path.cwd()))
        self.btn_browse = QPushButton("Browse")
        self.btn_browse.clicked.connect(self._browse_save)
        h.addWidget(self.save_edit,1); h.addWidget(self.btn_browse)
        form.addRow("Output folder:", row)

        row = QWidget(); h=QHBoxLayout(row); h.setContentsMargins(0,0,0,0); h.setSpacing(8)
        self.daq_combo = QComboBox(); self.daq_combo.setMinimumWidth(140)
        self.baud_spin = make_spinbox(9600, 921600, 115200, 9600)
        self.btn_refresh = QPushButton("Refresh")
        self.btn_refresh.clicked.connect(self._refresh_ports)
        h.addWidget(QLabel("COM")); h.addWidget(self.daq_combo)
        h.addWidget(QLabel("Baud")); h.addWidget(self.baud_spin)
        h.addWidget(self.btn_refresh); h.addStretch(1)
        form.addRow("Old board:", row)

        row = QWidget(); h=QHBoxLayout(row); h.setContentsMargins(0,0,0,0); h.setSpacing(8)
        self.btn_animals = QPushButton("Animal details")
        self.btn_layout = QPushButton("EcoHAB layout")
        self.btn_animals.clicked.connect(self._open_animals)
        self.btn_layout.clicked.connect(self._open_layout)
        h.addWidget(self.btn_animals); h.addWidget(self.btn_layout); h.addStretch(1)
        form.addRow("Setup:", row)

        row = QWidget(); h=QHBoxLayout(row); h.setContentsMargins(0,0,0,0); h.setSpacing(8)
        self.btn_start = QPushButton("START")
        self.btn_stop = QPushButton("STOP")
        self.btn_stop.setEnabled(False)
        self.btn_start.clicked.connect(self._on_start)
        self.btn_stop.clicked.connect(self._on_stop)
        h.addWidget(self.btn_start); h.addWidget(self.btn_stop); h.addStretch(1)
        form.addRow("Run:", row)

        self.lbl_daq = QLabel("DAQ: idle")
        self.lbl_file = QLabel("File: -")
        self.lbl_file.setTextInteractionFlags(Qt.TextSelectableByMouse)
        form.addRow("Status:", self.lbl_daq)
        form.addRow("Output:", self.lbl_file)

        self.tags_box = QGroupBox("Tag totals (refreshes every 60 seconds)")
        self.tags_box.setFixedHeight(320)
        self.tags_box.setMinimumWidth(420)
        tv = QVBoxLayout(self.tags_box)
        tv.setContentsMargins(8,8,8,8)
        tv.setSpacing(6)
        self.tags_info = QLabel("Tag totals: not yet refreshed")
        tv.addWidget(self.tags_info)
        self.tag_table = QTableWidget(0,2)
        self.tag_table.setHorizontalHeaderLabels(["TAG", "Count"])
        self.tag_table.setEditTriggers(QTableWidget.NoEditTriggers)
        self.tag_table.setSelectionBehavior(QTableWidget.SelectRows)
        self.tag_table.setColumnWidth(0, 240)
        self.tag_table.setColumnWidth(1, 90)
        tv.addWidget(self.tag_table,1)
        top.addWidget(self.tags_box,2)

        self.activity_box = QGroupBox("RFID antenna activity")
        self.activity_box.setStyleSheet("QGroupBox { background:#151a23; } QLabel { background: transparent; } QWidget { background: transparent; }")
        main_v.addWidget(self.activity_box, 1)
        av = QVBoxLayout(self.activity_box)
        grid = QGridLayout()
        grid.setSpacing(10)
        self.rfid_leds: Dict[str, QLabel] = {}
        self.rfid_tag_labels: Dict[str, QLabel] = {}
        for idx, ch in enumerate(RFID_CHANNELS):
            w = QWidget()
            w.setStyleSheet("background: transparent;")
            wh = QHBoxLayout(w); wh.setContentsMargins(6,4,6,4); wh.setSpacing(6)
            led = QLabel(); led.setStyleSheet(led_style(False))
            lab = QLabel(f"Antenna {ch}")
            lab.setStyleSheet("font-weight:600; background: transparent;")
            tag_lbl = QLabel("TAG: -")
            tag_lbl.setMinimumWidth(150)
            tag_lbl.setTextInteractionFlags(Qt.TextSelectableByMouse)
            tag_lbl.setStyleSheet("font-family: Consolas, monospace; background: transparent; color:#e6e6e6;")
            wh.addWidget(led)
            wh.addWidget(lab)
            wh.addWidget(tag_lbl)
            wh.addStretch(1)
            self.rfid_leds[ch] = led
            self.rfid_tag_labels[ch] = tag_lbl
            grid.addWidget(w, idx//4, idx%4)
        av.addLayout(grid)
        av.addStretch(1)

    def _browse_save(self):
        d = QFileDialog.getExistingDirectory(self, "Select output folder", self.save_edit.text().strip() or str(Path.cwd()))
        if d:
            self.save_edit.setText(d)

    def _refresh_ports(self):
        cur = self.daq_combo.currentText()
        ports = list_serial_ports()
        self.daq_combo.clear(); self.daq_combo.addItems(ports)
        if cur in ports:
            self.daq_combo.setCurrentText(cur)
        self.statusBar().showMessage(f"Detected {len(ports)} COM port(s)." if ports else "No COM ports detected.")

    def get_shared_config_paths(self) -> Tuple[Path, Path]:
        save_dir = Path(self.save_edit.text().strip() or str(Path.cwd()))
        save_dir.mkdir(parents=True, exist_ok=True)
        exp = safe_slug(self.exp_edit.text(), "old_board_experiment")
        return save_dir / f"{exp}_config.csv", save_dir / f"{exp}_config.json"

    def load_shared_config_from_file(self, json_path: Path) -> Optional[Dict[str, Any]]:
        cfg = read_config_json(json_path)
        if not cfg:
            return None
        if "experiment" not in cfg:
            cfg["experiment"] = {"name": self.exp_edit.text().strip(), "updated": dt.datetime.now().isoformat()}
        if "layout" not in cfg:
            cfg["layout"] = {"cages": {}, "tunnels": {}, "antenna_combinations_interp": {}, "antenna_combinations": {}, "tunnels_map": {}}
        if "animals" not in cfg:
            cfg["animals"] = {"n_mice": 0, "rows": {}}
        return cfg

    def update_shared_config(self, layout: Optional[Dict[str, Any]], animals: Optional[Dict[str, Any]], environment: Optional[Dict[str, Any]] = None, hardware_trigger: Optional[Dict[str, Any]] = None) -> None:
        csv_path, json_path = self.get_shared_config_paths()
        cfg = read_config_json(json_path) or new_config_skeleton(self.exp_edit.text().strip())
        cfg.setdefault("experiment", {})
        cfg.setdefault("layout", {})
        cfg.setdefault("animals", {"n_mice": 0, "rows": {}})
        cfg["experiment"]["name"] = self.exp_edit.text().strip()
        cfg["experiment"]["updated"] = dt.datetime.now().isoformat()
        cfg["experiment"].setdefault("board_type", "old_rfid_board")
        if getattr(self, "_experiment_started_iso", ""):
            cfg["experiment"]["start_datetime"] = self._experiment_started_iso
        if layout is not None:
            cfg["layout"] = normalize_layout_for_config(layout)
        else:
            cfg["layout"] = normalize_layout_for_config(cfg.get("layout") or {})
        if animals is not None:
            cfg["animals"] = normalize_animals_for_config(animals)
        else:
            cfg["animals"] = normalize_animals_for_config(cfg.get("animals") or {})
        write_config_json(cfg, json_path)
        write_config_csv(cfg, csv_path)

    def _open_animals(self):
        AnimalDetailsDialog(self).exec()

    def _open_layout(self):
        LayoutDialog(self).exec()

    def _on_start(self):
        port = self.daq_combo.currentText().strip()
        if not port:
            QMessageBox.critical(self, "COM missing", "Select the old-board COM port.")
            return
        save_dir = self.save_edit.text().strip() or str(Path.cwd())
        exp = safe_slug(self.exp_edit.text(), "old_board_experiment")
        stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
        prefix = f"{exp}_{stamp}"
        self.current_prefix = prefix
        self._experiment_started_iso = dt.datetime.now().isoformat()
        try:
            self.board = OldBoardAcquisition(port=port, save_dir=save_dir, file_prefix=prefix, baudrate=int(self.baud_spin.value()))
            self.board.start()
            self.update_shared_config(layout=None, animals=None)
            self.btn_start.setEnabled(False); self.btn_stop.setEnabled(True)
            self.lbl_file.setText(str(self.board.data_file))
            self.statusBar().showMessage(f"Old board acquisition started on {port}.")
        except Exception as e:
            QMessageBox.critical(self, "Start failed", str(e))

    def _on_stop(self):
        try:
            if self.board:
                self.board.stop()
        except Exception:
            pass
        self.btn_start.setEnabled(True); self.btn_stop.setEnabled(False)
        self.statusBar().showMessage("Old board acquisition stopped.")

    def _tick(self):
        if self.board:
            active, counts, last_tags = self.board.get_status_snapshot()
            for ch, led in self.rfid_leds.items():
                is_active = bool(active.get(ch, False))
                led.setStyleSheet(led_style(is_active))
                if ch in self.rfid_tag_labels:
                    tag = last_tags.get(ch, "")
                    self.rfid_tag_labels[ch].setText(f"TAG: {tag if tag else '-'}")
            self.lbl_daq.setText(f"DAQ: ok={self.board.lines_ok} bad={self.board.lines_bad} bytes={self.board.bytes_read} | {Path(self.board.data_file).name}")
            now = time.monotonic()
            if now - self._last_tag_table_refresh >= 60 or self._last_tag_table_refresh == 0:
                self._refresh_tag_table(counts)
                self._last_tag_table_refresh = now
        else:
            for ch, led in getattr(self, "rfid_leds", {}).items():
                led.setStyleSheet(led_style(False))
                if hasattr(self, "rfid_tag_labels") and ch in self.rfid_tag_labels:
                    self.rfid_tag_labels[ch].setText("TAG: -")
            self.lbl_daq.setText("DAQ: idle")

    def _refresh_tag_table(self, counts: Optional[Dict[str,int]] = None):
        counts = counts if counts is not None else (self.board.get_unique_tags_snapshot() if self.board else {})
        items = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
        self.tag_table.setRowCount(len(items))
        for r, (tag, count) in enumerate(items):
            self.tag_table.setItem(r, 0, QTableWidgetItem(str(tag)))
            self.tag_table.setItem(r, 1, QTableWidgetItem(str(count)))
        self.tags_info.setText(f"Unique tags: {len(items)} | total events: {sum(counts.values()) if counts else 0}")

    def closeEvent(self, event):
        try:
            if self.board:
                self.board.stop()
        except Exception:
            pass
        super().closeEvent(event)


if __name__ == "__main__":
    app = QApplication.instance() or QApplication([])
    apply_dark_theme(app)
    win = MainWindow()
    win.show()
    app.exec()

